# Colab GPU pipeline — Final Clean Generalization Protocol

This notebook implements the final experiment design:

1. Create one deterministic train/val/test split from `full_curated_v1`, stratified by `subset`.
2. Train YOLO/RT-DETR on the **training split containing all three training conditions**:
   - `synthetic_drone_positive`
   - `synthetic_distractor_only` / hard negatives
   - `synthetic_drone_plus_distractor` / mixed
3. Evaluate every model separately on the held-out test subsets:
   - `test_drone_positive`
   - `test_hard_negative`
   - `test_mixed`
4. Evaluate GroundingDINO zero-shot on the same test subsets.
5. Save one combined table with `eval_subset` and model metrics.

Important: the hard-negative subset is used as negative training data together with positive drone examples. We do **not** train a standalone detector only on hard negatives.


## Important runtime notes

- Choose **L4 GPU** or **A100 GPU** before running RT-DETR-L. The notebook now stops early if RT-DETR training is enabled on a slow GPU such as T4.
- Training commands now use live streaming output, so epoch progress should appear while the cell is running.
- Training images/labels are copied to local Colab SSD (`/content/...`) by default for speed, but model outputs still save under your Google Drive project folder: `outputs/training/...`. This keeps `.pt` weights safe if the Colab session closes.


In [ ]:
# 1) Configuration, Drive mount, GPU check, helpers
from pathlib import Path
import os
import sys
import subprocess
import shutil
import time
from datetime import datetime

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
except Exception as e:
    print("Drive mount failed or not running in Colab:", repr(e))

# Your current Drive project path
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification"

PROJECT_ROOT = Path(DRIVE_PROJECT_PATH)
DATASET_ROOT = PROJECT_ROOT / "data/synthetic/full_curated_v1"

# Final clean generalization protocol:
# Train on train split containing positives + hard negatives + mixed.
# Evaluate separately on held-out test_drone_positive, test_hard_negative, test_mixed.
RUN_MODE = "final_clean"

# For final clean, use all split rows by default.
# Set TRAIN_MAX_IMAGES to an integer only for quick debugging.
TRAIN_MAX_IMAGES = None
EVAL_MAX_IMAGES_PER_SUBSET = None

EPOCHS_YOLO = 30
EPOCHS_RTDETR = 5

SPLIT_NAME = "full_curated_v1_final_clean_split"
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

BATCH_YOLO = 16
BATCH_RTDETR = 2  # good for L4; use 1 on T4 if you intentionally train there
IMGSZ = 640

# Speed/stability options.
# Copies training images/labels from Google Drive to Colab's local SSD (/content) before training.
# Training outputs still save to Google Drive under PROJECT_ROOT/outputs/training/...
USE_LOCAL_TRAINING_DATA = True
LOCAL_TRAINING_ROOT = Path("/content/drone_final_clean_training") / SPLIT_NAME

# Safety guard: RT-DETR-L is too slow on T4. Leave True to avoid accidentally repeating the T4 issue.
REQUIRE_FAST_GPU_FOR_RTDETR = True
FAST_GPU_KEYWORDS = ["L4", "A100", "H100"]

BASE_EVAL_CONFIG = "configs/eval_colab_full_curated_v1.yaml"
EVAL_CONFIG_DIR = PROJECT_ROOT / "configs/generated_eval_subsets"
EVAL_OUTPUT_ROOT = PROJECT_ROOT / "outputs/evaluation/final_clean_full_curated_v1"

# Stable run names. The eval config should point to these stable folders.
YOLO_RUN_NAME = "yolo11n_drone_colab"
RTDETR_RUN_NAME = "rtdetr_l_drone_colab"

RUN_GDINO = True
RUN_YOLO_TRAINING = True
RUN_RTDETR_TRAINING = True
RUN_YOLO_INFERENCE = True
RUN_RTDETR_INFERENCE = True

# Set True only when you intentionally want to delete previous training/prediction outputs.
RESET_MODEL_OUTPUTS = False
RESET_PREDICTIONS = True

# Evaluation subsets derived from held-out test rows.
EVAL_SUBSETS = {
    "test_drone_positive": "synthetic_drone_positive",
    "test_hard_negative": "synthetic_distractor_only",
    "test_mixed": "synthetic_drone_plus_distractor",
}

def list_training_runs(root, pattern):
    """Return real Ultralytics run folders that contain weights/best.pt.

    Excludes backup folders created by this notebook, because their names can also match
    patterns like `yolo11n_drone_colab*` and can otherwise be accidentally selected
    as the latest run.
    """
    root = Path(root)
    runs = []
    for p in root.glob(pattern):
        if not p.is_dir():
            continue
        name = p.name.lower()
        if "backup" in name or "old_before_latest_sync" in name or "disabled" in name:
            continue
        if (p / "weights/best.pt").exists():
            runs.append(p)
    return sorted(runs)

def get_latest_training_run(root, pattern):
    """Return the latest real Ultralytics run folder by modification time."""
    runs = list_training_runs(root, pattern)
    if not runs:
        return None
    return max(runs, key=lambda p: p.stat().st_mtime)

def sync_latest_run_to_stable(root, pattern, stable_name):
    """Copy latest Ultralytics run, including suffix folders like -2/-3, to stable_name.

    Why this is needed:
    Ultralytics avoids overwriting existing runs. If `name=yolo11n_drone_colab` already exists,
    a new training run may be saved as `yolo11n_drone_colab-2`, `-3`, etc.
    The evaluation config points to the stable folder `yolo11n_drone_colab`, so this sync step
    prevents evaluation from accidentally using an old checkpoint.
    """
    root = Path(root)
    latest = get_latest_training_run(root, pattern)
    if latest is None:
        print(f"No training runs found for {pattern} under {root}")
        return None

    stable = root / stable_name
    if latest.resolve() == stable.resolve():
        print(f"Stable folder is already latest: {stable}")
        return stable

    backup = root / f"{stable_name}_old_before_latest_sync"
    if backup.exists():
        shutil.rmtree(backup)
    if stable.exists():
        shutil.move(str(stable), str(backup))
        print(f"Backed up old stable folder to: {backup}")

    shutil.copytree(latest, stable)
    print(f"Synced latest run to stable folder:")
    print(f"  latest: {latest}")
    print(f"  stable: {stable}")
    return stable

def format_cmd(cmd):
    return " ".join([repr(str(x)) if " " in str(x) else str(x) for x in cmd])

def run_command(cmd, check=True, cwd=None):
    """Run a command and stream stdout/stderr live.

    This avoids the previous black-box behavior where subprocess.run(capture_output=True)
    hid the training logs until the command ended. It also sets PYTHONUNBUFFERED=1 so
    Python training scripts flush progress sooner.
    """
    cmd = [str(x) for x in cmd]
    print("\nRunning:")
    print(format_cmd(cmd))
    print("\n----- live output starts -----", flush=True)

    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"

    start = time.time()
    process = subprocess.Popen(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        cwd=str(cwd) if cwd is not None else None,
        env=env,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)

    process.wait()
    elapsed_min = (time.time() - start) / 60
    print("\n----- live output ended -----")
    print(f"Return code: {process.returncode}")
    print(f"Elapsed time: {elapsed_min:.1f} minutes")

    if check and process.returncode != 0:
        raise RuntimeError("Command failed. See live output above.")
    return process.returncode

print("CUDA check:")
GPU_NAME = ""
try:
    import torch
    print("CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
        print("GPU:", GPU_NAME)
except Exception as e:
    print("Torch check failed:", repr(e))

if RUN_RTDETR_TRAINING and REQUIRE_FAST_GPU_FOR_RTDETR:
    if not any(k.lower() in GPU_NAME.lower() for k in FAST_GPU_KEYWORDS):
        raise RuntimeError(
            f"RT-DETR training is enabled, but the current GPU is '{GPU_NAME}'.\n"
            f"Choose L4/A100/H100 or set RUN_RTDETR_TRAINING=False.\n"
            f"This guard prevents another silent multi-hour T4 run."
        )

print("\nRun configuration:")
print("RUN_MODE:", RUN_MODE)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("USE_LOCAL_TRAINING_DATA:", USE_LOCAL_TRAINING_DATA)
print("LOCAL_TRAINING_ROOT:", LOCAL_TRAINING_ROOT)
print("SPLIT_NAME:", SPLIT_NAME)
print("TRAIN_MAX_IMAGES:", TRAIN_MAX_IMAGES)
print("EVAL_MAX_IMAGES_PER_SUBSET:", EVAL_MAX_IMAGES_PER_SUBSET)
print("EPOCHS_YOLO:", EPOCHS_YOLO)
print("EPOCHS_RTDETR:", EPOCHS_RTDETR)
print("EVAL_OUTPUT_ROOT:", EVAL_OUTPUT_ROOT)


Mounted at /content/drive
CUDA check:
CUDA: True
GPU: NVIDIA A100-SXM4-80GB

Run configuration:
RUN_MODE: final_clean
PROJECT_ROOT: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification
DATASET_ROOT: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/synthetic/full_curated_v1
USE_LOCAL_TRAINING_DATA: True
LOCAL_TRAINING_ROOT: /content/drone_final_clean_training/full_curated_v1_final_clean_split
SPLIT_NAME: full_curated_v1_final_clean_split
TRAIN_MAX_IMAGES: None
EVAL_MAX_IMAGES_PER_SUBSET: None
EPOCHS_YOLO: 30
EPOCHS_RTDETR: 5
EVAL_OUTPUT_ROOT: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1


In [ ]:
# 2) Verify project and dataset paths
required_paths = [
    PROJECT_ROOT,
    PROJECT_ROOT / "code" / "scripts",
    PROJECT_ROOT / "code" / "src",
    PROJECT_ROOT / "configs",
    PROJECT_ROOT / "requirements-colab.txt",
    DATASET_ROOT / "images",
    DATASET_ROOT / "labels",
    DATASET_ROOT / "metadata.csv",
    PROJECT_ROOT / BASE_EVAL_CONFIG,
]

print("Path checks:")
missing = []
for p in required_paths:
    ok = p.exists()
    print(f"{ok!s:5}  {p}")
    if not ok:
        missing.append(str(p))

if missing:
    raise FileNotFoundError("Missing required paths:\n" + "\n".join(missing))

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "code" / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "code" / "src"))

print("\nProject and dataset are ready.")


Path checks:
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/scripts
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/src
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/requirements-colab.txt
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/synthetic/full_curated_v1/images
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/synthetic/full_curated_v1/labels
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/synthetic/full_curated_v1/metadata.csv
True   /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs/eval_colab_full

In [ ]:
# #Now verify that the local /content/... copy exists and that DATA_YAML points to it.
# from pathlib import Path
# import os
# import yaml

# local_root = Path("/content/drone_final_clean_training")

# print("local_root exists:", local_root.exists())
# print("local_root:", local_root)

# print("\nTop-level contents:")
# if local_root.exists():
#     for p in sorted(local_root.iterdir()):
#         print(" -", p)

# split_root = Path("/content/drone_final_clean_training/full_curated_v1_final_clean_split")
# print("\nsplit_root exists:", split_root.exists())
# print("split_root:", split_root)

# print("\nRecursive quick check:")
# for sub in [
#     "images/train",
#     "images/val",
#     "labels/train",
#     "labels/val",
# ]:
#     p = split_root / sub
#     print(f"{sub}: exists={p.exists()}, files={len(list(p.glob('*'))) if p.exists() else 'NA'}")

# yaml_path = split_root / "dataset.yaml"
# print("\ndataset.yaml exists:", yaml_path.exists())
# print("dataset.yaml:", yaml_path)

# if yaml_path.exists():
#     with open(yaml_path, "r") as f:
#         data_yaml = yaml.safe_load(f)
#     print("\ndataset.yaml contents:")
#     print(data_yaml)

#     print("\nImportant path check:")
#     print("path:", data_yaml.get("path"))
#     print("train:", data_yaml.get("train"))
#     print("val:", data_yaml.get("val"))

#     if str(data_yaml.get("path", "")).startswith("/content/"):
#         print("\n✅ GOOD: dataset.yaml points to local Colab storage.")
#     else:
#         print("\n⚠️ WARNING: dataset.yaml does not point to local /content storage.")

local_root exists: False
local_root: /content/drone_final_clean_training

Top-level contents:

split_root exists: False
split_root: /content/drone_final_clean_training/full_curated_v1_final_clean_split

Recursive quick check:
images/train: exists=False, files=NA
images/val: exists=False, files=NA
labels/train: exists=False, files=NA
labels/val: exists=False, files=NA

dataset.yaml exists: False
dataset.yaml: /content/drone_final_clean_training/full_curated_v1_final_clean_split/dataset.yaml


In [ ]:
# 3) Optional cleanup of old outputs
# RESET_MODEL_OUTPUTS=False by default, because training takes time.
# If you set it to True, this removes all matching Ultralytics runs, including -2/-3 suffix folders.
if RESET_MODEL_OUTPUTS:
    cleanup_patterns = [
        (PROJECT_ROOT / "outputs/training/yolo", f"{YOLO_RUN_NAME}*"),
        (PROJECT_ROOT / "outputs/training/rtdetr", f"{RTDETR_RUN_NAME}*"),
    ]
    for root, pattern in cleanup_patterns:
        for p in root.glob(pattern):
            print("Removing training run:", p)
            shutil.rmtree(p, ignore_errors=True)

if RESET_PREDICTIONS:
    # Final-clean predictions live under separate output folders for each test subset.
    if EVAL_OUTPUT_ROOT.exists():
        for p in EVAL_OUTPUT_ROOT.glob("*/predictions/*_predictions.csv"):
            print("Removing prediction:", p)
            p.unlink()

print("Cleanup step complete.")


Cleanup step complete.


In [ ]:
# 4) Install dependencies + GroundingDINO
# This cell can take several minutes.
os.chdir(PROJECT_ROOT)

run_command([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements-colab.txt")])
run_command([sys.executable, "-m", "pip", "install", "-q", "transformers==4.41.2", "tokenizers<0.20"])

(PROJECT_ROOT / "external").mkdir(exist_ok=True)
(PROJECT_ROOT / "weights/groundingdino").mkdir(parents=True, exist_ok=True)

gdino_dir = PROJECT_ROOT / "external/GroundingDINO"
if not gdino_dir.is_dir():
    run_command(["git", "clone", "https://github.com/IDEA-Research/GroundingDINO.git", str(gdino_dir)])

ckpt = PROJECT_ROOT / "weights/groundingdino/groundingdino_swint_ogc.pth"
if not ckpt.is_file():
    run_command([
        "wget", "-q", "-O", str(ckpt),
        "https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth",
    ])

# Patch for newer torch/CUDA builds when needed.
cuda_file = gdino_dir / "groundingdino/models/GroundingDINO/csrc/MsDeformAttn/ms_deform_attn_cuda.cu"
if cuda_file.exists():
    text = cuda_file.read_text()
    text = text.replace("value.type()", "value.scalar_type()")
    text = text.replace("value.scalar_type().is_cuda()", "value.is_cuda()")
    cuda_file.write_text(text)

run_command([sys.executable, "-m", "pip", "install", "--no-build-isolation", "-e", str(gdino_dir)])

# Verify pinned transformers version
import transformers
print("transformers:", transformers.__version__)



Running:
/usr/bin/python3 -m pip install -q -r '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/requirements-colab.txt'

----- live output starts -----
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 12.8 MB/s eta 0:00:00

----- live output ended -----
Return code: 0
Elapsed time: 0.1 minutes

Running:
/usr/bin/python3 -m pip install -q transformers==4.41.2 tokenizers<0.20

----- live output starts -----
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.1 MB/s

In [ ]:
# 5) Verify imports
os.chdir(PROJECT_ROOT)

gdino_path = str(PROJECT_ROOT / "external/GroundingDINO")
src_path = str(PROJECT_ROOT / "code" / "src")
for p in [gdino_path, src_path]:
    if p not in sys.path:
        sys.path.insert(0, p)

import groundingdino
print("GroundingDINO import OK")

from ultralytics import YOLO
print("Ultralytics import OK")


GroundingDINO import OK
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics import OK


In [ ]:
# 6) Create final-clean train / val / test split and Ultralytics dataset.yaml
# Training split contains positives + hard negatives + mixed.
# Evaluation will later be done separately on test_drone_positive, test_hard_negative, test_mixed.

import pandas as pd
import random
from pathlib import Path

metadata_path = DATASET_ROOT / "metadata.csv"
metadata_df = pd.read_csv(metadata_path)

required_cols = {"image_id", "subset"}
missing_cols = required_cols - set(metadata_df.columns)
if missing_cols:
    raise ValueError(f"metadata.csv missing required columns: {missing_cols}")

split_root = PROJECT_ROOT / "data/training" / SPLIT_NAME
split_root.mkdir(parents=True, exist_ok=True)

# Optional cap for quick debugging, stratified by subset.
work_df = metadata_df.copy()
if TRAIN_MAX_IMAGES is not None and TRAIN_MAX_IMAGES < len(work_df):
    parts = []
    per_subset = max(1, TRAIN_MAX_IMAGES // work_df["subset"].nunique())
    for subset, g in work_df.groupby("subset"):
        g = g.sample(frac=1.0, random_state=42)
        parts.append(g.head(per_subset))
    work_df = pd.concat(parts, ignore_index=True)
    if len(work_df) > TRAIN_MAX_IMAGES:
        work_df = work_df.sample(n=TRAIN_MAX_IMAGES, random_state=42).reset_index(drop=True)

# Deterministic split inside every subset.
split_rows = []
rng = random.Random(42)

for subset, g in work_df.groupby("subset"):
    ids = list(g.index)
    rng.shuffle(ids)

    n = len(ids)
    n_train = int(round(n * TRAIN_RATIO))
    n_val = int(round(n * VAL_RATIO))

    train_ids = set(ids[:n_train])
    val_ids = set(ids[n_train:n_train + n_val])

    for idx in ids:
        row = work_df.loc[idx].copy()
        if idx in train_ids:
            row["split_final_clean"] = "train"
        elif idx in val_ids:
            row["split_final_clean"] = "val"
        else:
            row["split_final_clean"] = "test"
        split_rows.append(row)

split_df = pd.DataFrame(split_rows).reset_index(drop=True)

split_csv = split_root / "split_metadata.csv"
split_df.to_csv(split_csv, index=False)

print("Wrote split metadata:", split_csv)
print("\nSplit counts by subset:")
display(pd.crosstab(split_df["subset"], split_df["split_final_clean"]))

# Optionally copy all training/validation/test images and labels to local Colab SSD.
# This avoids slow training caused by repeatedly reading thousands of small files from mounted Google Drive.
if USE_LOCAL_TRAINING_DATA:
    TRAIN_DATASET_ROOT = LOCAL_TRAINING_ROOT
    (TRAIN_DATASET_ROOT / "images").mkdir(parents=True, exist_ok=True)
    (TRAIN_DATASET_ROOT / "labels").mkdir(parents=True, exist_ok=True)

    print("\nCopying training split files to local Colab SSD:")
    print("  from:", DATASET_ROOT)
    print("  to:  ", TRAIN_DATASET_ROOT)
    print("This may take a few minutes, but training should be faster and more stable afterward.")

    copied_images = 0
    copied_labels = 0
    created_empty_labels = 0

    for i, image_id in enumerate(split_df["image_id"].astype(str), start=1):
        src_img = DATASET_ROOT / "images" / image_id
        dst_img = TRAIN_DATASET_ROOT / "images" / image_id
        if not src_img.exists():
            raise FileNotFoundError(f"Missing image: {src_img}")
        if (not dst_img.exists()) or (dst_img.stat().st_size != src_img.stat().st_size):
            dst_img.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_img, dst_img)
            copied_images += 1

        label_name = Path(image_id).with_suffix(".txt").name
        src_lbl = DATASET_ROOT / "labels" / label_name
        dst_lbl = TRAIN_DATASET_ROOT / "labels" / label_name
        if src_lbl.exists():
            if (not dst_lbl.exists()) or (dst_lbl.stat().st_size != src_lbl.stat().st_size):
                dst_lbl.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_lbl, dst_lbl)
                copied_labels += 1
        else:
            # Ultralytics supports empty label files for negative images.
            if not dst_lbl.exists():
                dst_lbl.parent.mkdir(parents=True, exist_ok=True)
                dst_lbl.write_text("", encoding="utf-8")
                created_empty_labels += 1

        if i % 500 == 0 or i == len(split_df):
            print(f"  processed {i}/{len(split_df)} files...", flush=True)

    print(f"Local copy complete. copied_images={copied_images}, copied_labels={copied_labels}, created_empty_labels={created_empty_labels}")
else:
    TRAIN_DATASET_ROOT = DATASET_ROOT
    print("\nUsing training data directly from Google Drive:", TRAIN_DATASET_ROOT)

# Create image list files for Ultralytics.
# Labels are inferred from image paths by replacing /images/ with /labels/.
def image_path_for_id(image_id):
    return TRAIN_DATASET_ROOT / "images" / str(image_id)

for split_name in ["train", "val", "test"]:
    rows = split_df[split_df["split_final_clean"] == split_name]
    list_path = split_root / f"{split_name}_images.txt"
    with open(list_path, "w", encoding="utf-8") as f:
        for image_id in rows["image_id"].astype(str):
            p = image_path_for_id(image_id)
            if not p.exists():
                raise FileNotFoundError(f"Missing image: {p}")
            f.write(str(p) + "\n")
    print(f"{split_name}: {len(rows)} images -> {list_path}")

# Dataset YAML for Ultralytics training.
# Even when TRAIN_DATASET_ROOT is local /content, outputs still save to Drive because the training scripts
# are launched from PROJECT_ROOT and use PROJECT_ROOT/outputs/training/... as their project directory.
DATA_YAML = split_root / "dataset.yaml"
yaml_text = f"""# Auto-generated final clean split
path: "{TRAIN_DATASET_ROOT}"
train: "{split_root / 'train_images.txt'}"
val: "{split_root / 'val_images.txt'}"
test: "{split_root / 'test_images.txt'}"
nc: 1
names:
  0: drone
"""
DATA_YAML.write_text(yaml_text, encoding="utf-8")

print("\nDATA_YAML:", DATA_YAML)
print(DATA_YAML.read_text())

# Basic label sanity.
for split_name in ["train", "val", "test"]:
    rows = split_df[split_df["split_final_clean"] == split_name]
    non_empty = 0
    missing = 0
    for image_id in rows["image_id"].astype(str):
        label_path = TRAIN_DATASET_ROOT / "labels" / Path(image_id).with_suffix(".txt").name
        if not label_path.exists():
            missing += 1
        elif label_path.read_text(encoding="utf-8").strip():
            non_empty += 1
    print(f"{split_name}: rows={len(rows)}, non_empty_drone_labels={non_empty}, missing_labels={missing}")


Wrote split metadata: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/training/full_curated_v1_final_clean_split/split_metadata.csv

Split counts by subset:


split_final_clean,test,train,val
subset,,,
synthetic_distractor_only,180,840,180
synthetic_drone_plus_distractor,180,840,180
synthetic_drone_positive,180,840,180



Copying training split files to local Colab SSD:
  from: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/synthetic/full_curated_v1
  to:   /content/drone_final_clean_training/full_curated_v1_final_clean_split
This may take a few minutes, but training should be faster and more stable afterward.
  processed 500/3600 files...
  processed 1000/3600 files...
  processed 1500/3600 files...
  processed 2000/3600 files...
  processed 2500/3600 files...
  processed 3000/3600 files...
  processed 3500/3600 files...
  processed 3600/3600 files...
Local copy complete. copied_images=3600, copied_labels=3600, created_empty_labels=0
train: 2520 images -> /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/training/full_curated_v1_final_clean_split/train_images.txt
val: 540 images -> /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/training/full_curated_v1_final_clean_split/val_images.txt
t

In [ ]:
from pathlib import Path

for txt_name in ["train_images.txt", "val_images.txt", "test_images.txt"]:
    txt_path = Path("/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/training/full_curated_v1_final_clean_split") / txt_name
    print("\n", txt_name)
    print("exists:", txt_path.exists())
    lines = txt_path.read_text().splitlines()[:5]
    for line in lines:
        print(line)


 train_images.txt
exists: True
/content/drone_final_clean_training/full_curated_v1_final_clean_split/images/img_001334.png
/content/drone_final_clean_training/full_curated_v1_final_clean_split/images/img_002505.png
/content/drone_final_clean_training/full_curated_v1_final_clean_split/images/img_001497.png
/content/drone_final_clean_training/full_curated_v1_final_clean_split/images/img_001295.png
/content/drone_final_clean_training/full_curated_v1_final_clean_split/images/img_002386.png

 val_images.txt
exists: True
/content/drone_final_clean_training/full_curated_v1_final_clean_split/images/img_002938.png
/content/drone_final_clean_training/full_curated_v1_final_clean_split/images/img_002986.png
/content/drone_final_clean_training/full_curated_v1_final_clean_split/images/img_001880.png
/content/drone_final_clean_training/full_curated_v1_final_clean_split/images/img_000936.png
/content/drone_final_clean_training/full_curated_v1_final_clean_split/images/img_002163.png

 test_images.txt


In [ ]:
#rewriting the train/val/test .txt files locally so the whole training input is local.
from pathlib import Path
import yaml

LOCAL_SPLIT_ROOT = Path("/content/drone_final_clean_training/full_curated_v1_final_clean_split")
DRIVE_SPLIT_ROOT = Path("/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/training/full_curated_v1_final_clean_split")

for split in ["train", "val", "test"]:
    drive_txt = DRIVE_SPLIT_ROOT / f"{split}_images.txt"
    local_txt = LOCAL_SPLIT_ROOT / f"{split}_images.txt"

    lines = drive_txt.read_text().splitlines()
    new_lines = []

    for line in lines:
        p = Path(line)
        # Keep only the filename and map it to local images/split/
        new_lines.append(str(LOCAL_SPLIT_ROOT / "images" / split / p.name))

    local_txt.write_text("\n".join(new_lines) + "\n")
    print(f"Wrote local {split} txt:", local_txt, "rows:", len(new_lines))

yaml_path = LOCAL_SPLIT_ROOT / "dataset.yaml"

data_yaml = {
    "path": str(LOCAL_SPLIT_ROOT),
    "train": str(LOCAL_SPLIT_ROOT / "train_images.txt"),
    "val": str(LOCAL_SPLIT_ROOT / "val_images.txt"),
    "test": str(LOCAL_SPLIT_ROOT / "test_images.txt"),
    "nc": 1,
    "names": {0: "drone"},
}

with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

DATA_YAML = yaml_path

print("\n✅ Local dataset.yaml ready:")
print(DATA_YAML)
print(yaml_path.read_text())

Wrote local train txt: /content/drone_final_clean_training/full_curated_v1_final_clean_split/train_images.txt rows: 2520
Wrote local val txt: /content/drone_final_clean_training/full_curated_v1_final_clean_split/val_images.txt rows: 540
Wrote local test txt: /content/drone_final_clean_training/full_curated_v1_final_clean_split/test_images.txt rows: 540

✅ Local dataset.yaml ready:
/content/drone_final_clean_training/full_curated_v1_final_clean_split/dataset.yaml
path: /content/drone_final_clean_training/full_curated_v1_final_clean_split
train: /content/drone_final_clean_training/full_curated_v1_final_clean_split/train_images.txt
val: /content/drone_final_clean_training/full_curated_v1_final_clean_split/val_images.txt
test: /content/drone_final_clean_training/full_curated_v1_final_clean_split/test_images.txt
nc: 1
names:
  0: drone



In [ ]:
#verify
print("DATA_YAML:", DATA_YAML)
print(Path(DATA_YAML).read_text())

DATA_YAML: /content/drone_final_clean_training/full_curated_v1_final_clean_split/dataset.yaml
path: /content/drone_final_clean_training/full_curated_v1_final_clean_split
train: /content/drone_final_clean_training/full_curated_v1_final_clean_split/train_images.txt
val: /content/drone_final_clean_training/full_curated_v1_final_clean_split/val_images.txt
test: /content/drone_final_clean_training/full_curated_v1_final_clean_split/test_images.txt
nc: 1
names:
  0: drone



In [ ]:
# 7) Create held-out evaluation subset datasets and generated eval configs
# Each eval subset contains only TEST rows from one condition:
# - test_drone_positive
# - test_hard_negative
# - test_mixed

import yaml
import pandas as pd
from pathlib import Path
import shutil
import os

split_root = PROJECT_ROOT / "data/training" / SPLIT_NAME
split_df = pd.read_csv(split_root / "split_metadata.csv")

eval_dataset_root = PROJECT_ROOT / "data/evaluation/final_clean_full_curated_v1"
eval_dataset_root.mkdir(parents=True, exist_ok=True)
EVAL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
EVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def copy_or_link(src, dst):
    """Try symlink first; fallback to copy. Google Drive sometimes behaves better with copies."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        return
    try:
        os.symlink(src, dst)
    except Exception:
        shutil.copy2(src, dst)

def replace_strings(obj, replacements):
    """Recursively replace string fragments in a loaded YAML object."""
    if isinstance(obj, dict):
        return {k: replace_strings(v, replacements) for k, v in obj.items()}
    if isinstance(obj, list):
        return [replace_strings(v, replacements) for v in obj]
    if isinstance(obj, str):
        s = obj
        for old, new in replacements.items():
            s = s.replace(old, new)
        return s
    return obj

def make_eval_subset_dataset(eval_name, subset_value):
    rows = split_df[
        (split_df["split_final_clean"] == "test") &
        (split_df["subset"] == subset_value)
    ].copy()

    if EVAL_MAX_IMAGES_PER_SUBSET is not None:
        rows = rows.head(EVAL_MAX_IMAGES_PER_SUBSET).copy()

    subset_root = eval_dataset_root / eval_name
    images_dir = subset_root / "images"
    labels_dir = subset_root / "labels"
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    for image_id in rows["image_id"].astype(str):
        src_img = DATASET_ROOT / "images" / image_id
        src_lbl = DATASET_ROOT / "labels" / Path(image_id).with_suffix(".txt").name
        dst_img = images_dir / image_id
        dst_lbl = labels_dir / Path(image_id).with_suffix(".txt").name

        if not src_img.exists():
            raise FileNotFoundError(src_img)
        copy_or_link(src_img, dst_img)

        # Hard negatives should have empty labels; if label is missing, create empty file.
        if src_lbl.exists():
            copy_or_link(src_lbl, dst_lbl)
        else:
            dst_lbl.write_text("", encoding="utf-8")

    rows.to_csv(subset_root / "metadata.csv", index=False)
    return subset_root, rows

def make_eval_config(eval_name, subset_root):
    base_path = PROJECT_ROOT / BASE_EVAL_CONFIG
    cfg = yaml.safe_load(base_path.read_text(encoding="utf-8"))

    # Keep paths relative to PROJECT_ROOT when possible, because scripts receive --project-root.
    rel_subset_root = subset_root.relative_to(PROJECT_ROOT).as_posix()
    rel_output_root = (EVAL_OUTPUT_ROOT / eval_name).relative_to(PROJECT_ROOT).as_posix()

    replacements = {
        "data/synthetic/full_curated_v1": rel_subset_root,
        "outputs/evaluation/colab_full_curated_v1": rel_output_root,
        "outputs/evaluation/full_curated_v1": rel_output_root,
    }
    cfg = replace_strings(cfg, replacements)

    # Patch common schema keys explicitly if present.
    if isinstance(cfg, dict):
        for key in ["dataset_root", "data_root", "root"]:
            if key in cfg and isinstance(cfg[key], str):
                cfg[key] = rel_subset_root
        if "dataset" in cfg and isinstance(cfg["dataset"], dict):
            for key in ["root", "dataset_root", "data_root"]:
                if key in cfg["dataset"]:
                    cfg["dataset"][key] = rel_subset_root
            for key in ["metadata", "metadata_path"]:
                if key in cfg["dataset"]:
                    cfg["dataset"][key] = f"{rel_subset_root}/metadata.csv"
        for key in ["output_dir", "output_root"]:
            if key in cfg:
                cfg[key] = rel_output_root
        if "evaluation" in cfg and isinstance(cfg["evaluation"], dict):
            for key in ["output_dir", "output_root"]:
                if key in cfg["evaluation"]:
                    cfg["evaluation"][key] = rel_output_root

    out_config = EVAL_CONFIG_DIR / f"eval_{eval_name}.yaml"
    out_config.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return out_config

EVAL_JOBS = {}

for eval_name, subset_value in EVAL_SUBSETS.items():
    subset_root, rows = make_eval_subset_dataset(eval_name, subset_value)
    cfg_path = make_eval_config(eval_name, subset_root)
    EVAL_JOBS[eval_name] = {
        "subset": subset_value,
        "dataset_root": subset_root,
        "config": cfg_path,
        "output_root": EVAL_OUTPUT_ROOT / eval_name,
        "n_images": len(rows),
        "n_target_present": int(rows["target_present"].astype(bool).sum()) if "target_present" in rows.columns else None,
    }

print("Created evaluation jobs:")
for name, job in EVAL_JOBS.items():
    print("\n", name)
    for k, v in job.items():
        print(" ", k, ":", v)

print("\nTest counts:")
display(pd.DataFrame([
    {"eval_subset": name, **{k: str(v) for k, v in job.items() if k != "config"}}
    for name, job in EVAL_JOBS.items()
]))

print("\nExample generated config:")
first_cfg = next(iter(EVAL_JOBS.values()))["config"]
print(first_cfg)
print(first_cfg.read_text()[:2000])


Created evaluation jobs:

 test_drone_positive
  subset : synthetic_drone_positive
  dataset_root : /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/evaluation/final_clean_full_curated_v1/test_drone_positive
  config : /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs/generated_eval_subsets/eval_test_drone_positive.yaml
  output_root : /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_drone_positive
  n_images : 180
  n_target_present : 180

 test_hard_negative
  subset : synthetic_distractor_only
  dataset_root : /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/evaluation/final_clean_full_curated_v1/test_hard_negative
  config : /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs/generated_eval_subsets/eval_test_hard_negative.yaml
  output_root : /con

,eval_subset,subset,dataset_root,output_root,n_images,n_target_present
0,test_drone_positive,synthetic_drone_positive,/content/drive/MyDrive/Synthetic Stress Testin...,/content/drive/MyDrive/Synthetic Stress Testin...,180,180
1,test_hard_negative,synthetic_distractor_only,/content/drive/MyDrive/Synthetic Stress Testin...,/content/drive/MyDrive/Synthetic Stress Testin...,180,0
2,test_mixed,synthetic_drone_plus_distractor,/content/drive/MyDrive/Synthetic Stress Testin...,/content/drive/MyDrive/Synthetic Stress Testin...,180,180



Example generated config:
/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs/generated_eval_subsets/eval_test_drone_positive.yaml
dataset:
  name: full_curated_v1
  root: data/evaluation/final_clean_full_curated_v1/test_drone_positive
  images_dir: data/evaluation/final_clean_full_curated_v1/test_drone_positive/images
  labels_dir: data/evaluation/final_clean_full_curated_v1/test_drone_positive/labels
  metadata_csv: data/evaluation/final_clean_full_curated_v1/test_drone_positive/metadata.csv
  image_width: 640
  image_height: 640
evaluation:
  iou_thresholds:
  - 0.25
  - 0.5
  confidence_thresholds:
  - 0.05
  - 0.1
  - 0.25
  - 0.5
  primary_iou_threshold: 0.5
  tiny_object_iou_threshold: 0.25
  target_class_name: drone
  drone_class_id: 0
  drone_like_classes:
  - drone
  - quadcopter
  - unmanned aerial vehicle
outputs:
  root: outputs/evaluation/final_clean_full_curated_v1/test_drone_positive
grounding_dino:
  repo_dir: external/GroundingDI

In [ ]:
#This will rebuild the local split folders properly:
from pathlib import Path
import shutil
import yaml
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification")

DATASET_ROOT = PROJECT_ROOT / "data/synthetic/full_curated_v1"
DRIVE_SPLIT_ROOT = PROJECT_ROOT / "data/training/full_curated_v1_final_clean_split"
LOCAL_SPLIT_ROOT = Path("/content/drone_final_clean_training/full_curated_v1_final_clean_split")

SRC_IMAGES = DATASET_ROOT / "images"
SRC_LABELS = DATASET_ROOT / "labels"
SPLIT_METADATA = DRIVE_SPLIT_ROOT / "split_metadata.csv"

print("DATASET_ROOT exists:", DATASET_ROOT.exists())
print("SRC_IMAGES exists:", SRC_IMAGES.exists())
print("SRC_LABELS exists:", SRC_LABELS.exists())
print("SPLIT_METADATA exists:", SPLIT_METADATA.exists())

if not SPLIT_METADATA.exists():
    raise FileNotFoundError(f"Missing split metadata: {SPLIT_METADATA}")

df = pd.read_csv(SPLIT_METADATA)

print("\nColumns:", list(df.columns))
print(df.head())

# Detect the image id column
image_col_candidates = ["image_id", "filename", "image_name"]
image_col = next((c for c in image_col_candidates if c in df.columns), None)
if image_col is None:
    raise ValueError(f"Could not find image id column. Columns: {list(df.columns)}")

split_col = "split_final_clean"
if split_col not in df.columns:
    raise ValueError(f"Missing {split_col}. Columns: {list(df.columns)}")

print("\nUsing image column:", image_col)
print("Using split column:", split_col)

# Clean and rebuild only the local split folder
if LOCAL_SPLIT_ROOT.exists():
    print("\nRemoving broken local split:", LOCAL_SPLIT_ROOT)
    shutil.rmtree(LOCAL_SPLIT_ROOT)

for split in ["train", "val", "test"]:
    (LOCAL_SPLIT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (LOCAL_SPLIT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

# Copy files into YOLO expected layout
counts = {"train": 0, "val": 0, "test": 0}
empty_labels = 0
missing_images = []

for _, row in df.iterrows():
    split = row[split_col]
    img_name = str(row[image_col])

    if split not in counts:
        continue

    src_img = SRC_IMAGES / img_name
    if not src_img.exists():
        missing_images.append(str(src_img))
        continue

    dst_img = LOCAL_SPLIT_ROOT / "images" / split / img_name
    shutil.copy2(src_img, dst_img)

    label_name = Path(img_name).with_suffix(".txt").name
    src_label = SRC_LABELS / label_name
    dst_label = LOCAL_SPLIT_ROOT / "labels" / split / label_name

    if src_label.exists():
        shutil.copy2(src_label, dst_label)
    else:
        dst_label.write_text("")
        empty_labels += 1

    counts[split] += 1

print("\nCopied counts:", counts)
print("Created empty labels:", empty_labels)
print("Missing images:", len(missing_images))
if missing_images[:5]:
    print("First missing images:", missing_images[:5])

if missing_images:
    raise FileNotFoundError("Some images were missing. Stop and inspect paths.")

# Write local train/val/test txt files
for split in ["train", "val", "test"]:
    image_paths = sorted((LOCAL_SPLIT_ROOT / "images" / split).glob("*.png"))
    txt_path = LOCAL_SPLIT_ROOT / f"{split}_images.txt"
    txt_path.write_text("\n".join(str(p) for p in image_paths) + "\n")
    print(f"{split}: wrote {len(image_paths)} paths to {txt_path}")

# Write local dataset.yaml
yaml_path = LOCAL_SPLIT_ROOT / "dataset.yaml"
data_yaml = {
    "path": str(LOCAL_SPLIT_ROOT),
    "train": str(LOCAL_SPLIT_ROOT / "train_images.txt"),
    "val": str(LOCAL_SPLIT_ROOT / "val_images.txt"),
    "test": str(LOCAL_SPLIT_ROOT / "test_images.txt"),
    "nc": 1,
    "names": {0: "drone"},
}

with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

# Remove old Ultralytics cache files if any
for cache_file in LOCAL_SPLIT_ROOT.rglob("*.cache"):
    print("Removing cache:", cache_file)
    cache_file.unlink()

DATA_YAML = yaml_path

print("\n✅ Rebuilt local YOLO dataset correctly.")
print("DATA_YAML:", DATA_YAML)
print(yaml_path.read_text())

# Final verification
for split in ["train", "val", "test"]:
    n_img = len(list((LOCAL_SPLIT_ROOT / "images" / split).glob("*.png")))
    n_lab = len(list((LOCAL_SPLIT_ROOT / "labels" / split).glob("*.txt")))
    print(f"{split}: images={n_img}, labels={n_lab}")

    sample_img = next((LOCAL_SPLIT_ROOT / "images" / split).glob("*.png"), None)
    if sample_img:
        sample_label = LOCAL_SPLIT_ROOT / "labels" / split / sample_img.with_suffix(".txt").name
        print("  sample image exists:", sample_img.exists(), sample_img)
        print("  sample label exists:", sample_label.exists(), sample_label)

DATASET_ROOT exists: True
SRC_IMAGES exists: True
SRC_LABELS exists: True
SPLIT_METADATA exists: True

Columns: ['image_id', 'split', 'subset', 'target_present', 'target_class', 'distractor_classes', 'background_source', 'foreground_asset_source', 'background_type', 'background_path', 'background_filename', 'object_size_px', 'distance_bin', 'gaussian_noise_sigma', 'blur_level', 'contrast_level', 'lighting', 'difficulty', 'target_bbox', 'target_asset_id', 'distractor_bboxes', 'distractor_asset_ids', 'pasted_object_summary', 'bbox', 'asset_id', 'seed', 'foreground_asset_path', 'foreground_asset_id', 'foreground_source_dataset', 'generation_seed', 'background_id', 'background_category', 'drone_present', 'drone_asset_id', 'drone_bbox_x', 'drone_bbox_y', 'drone_bbox_w', 'drone_bbox_h', 'drone_size_px', 'distractor_present', 'distractor_type', 'distractor_asset_id', 'distractor_bbox_x', 'distractor_bbox_y', 'distractor_bbox_w', 'distractor_bbox_h', 'distractor_size_px', 'noise_sigma', 'blur_

In [ ]:
from pathlib import Path

expected = Path("/content/drone_final_clean_training/full_curated_v1_final_clean_split/dataset.yaml")

print("DATA_YAML variable:", DATA_YAML)
print("Expected local path:", expected)
print("DATA_YAML exists:", Path(DATA_YAML).exists())
print("Is DATA_YAML local?", Path(DATA_YAML) == expected)

print("\nCurrent DATA_YAML contents:")
print(Path(DATA_YAML).read_text())

DATA_YAML variable: /content/drone_final_clean_training/full_curated_v1_final_clean_split/dataset.yaml
Expected local path: /content/drone_final_clean_training/full_curated_v1_final_clean_split/dataset.yaml
DATA_YAML exists: True
Is DATA_YAML local? True

Current DATA_YAML contents:
path: /content/drone_final_clean_training/full_curated_v1_final_clean_split
train: /content/drone_final_clean_training/full_curated_v1_final_clean_split/train_images.txt
val: /content/drone_final_clean_training/full_curated_v1_final_clean_split/val_images.txt
test: /content/drone_final_clean_training/full_curated_v1_final_clean_split/test_images.txt
nc: 1
names:
  0: drone



In [ ]:
# 8) Fine-tune YOLO11n
# Uses run_command(), which now streams logs live instead of buffering them until the end.
if RUN_YOLO_TRAINING:
    cmd = [
        sys.executable,
        "-u",
        "code/code/scripts/train/train_yolo.py",
        "--data", str(DATA_YAML),
        "--weights", "yolo11n.pt",
        "--epochs", str(EPOCHS_YOLO),
        "--imgsz", str(IMGSZ),
        "--batch", str(BATCH_YOLO),
        "--device", "0",
        "--name", YOLO_RUN_NAME,
        "--workers", "2",
    ]
    run_command(cmd)
else:
    print("Skipping YOLO training.")



Running:
/usr/bin/python3 -u scripts/train/train_yolo.py --data /content/drone_final_clean_training/full_curated_v1_final_clean_split/dataset.yaml --weights yolo11n.pt --epochs 30 --imgsz 640 --batch 16 --device 0 --name yolo11n_drone_colab --workers 2

----- live output starts -----
Training YOLO from yolo11n.pt
  data=/content/drone_final_clean_training/full_curated_v1_final_clean_split/dataset.yaml
  epochs=30 imgsz=640 batch=16 device=0
Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drone_final_clean_training/full_curated_v1_final_clean_split/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=Non

In [ ]:
# 9) Inspect YOLO training results
import pandas as pd

yolo_root = PROJECT_ROOT / "outputs/training/yolo"
stable_yolo = yolo_root / YOLO_RUN_NAME
latest_yolo = get_latest_training_run(yolo_root, f"{YOLO_RUN_NAME}*")

print("Stable YOLO folder:", stable_yolo)
print("Latest YOLO folder:", latest_yolo)

for label, run_dir in [("stable", stable_yolo), ("latest", latest_yolo)]:
    if run_dir is None:
        continue
    weights_best = run_dir / "weights/best.pt"
    results_csv = run_dir / "results.csv"
    print(f"\n{label.upper()} YOLO best weights:", weights_best, "exists:", weights_best.exists())
    print(f"{label.upper()} YOLO results.csv:", results_csv, "exists:", results_csv.exists())
    if results_csv.exists():
        yolo_results = pd.read_csv(results_csv)
        print(f"{label.upper()} YOLO epochs:", len(yolo_results))
        display(yolo_results.tail())

Stable YOLO folder: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab
Latest YOLO folder: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab-9

STABLE YOLO best weights: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab/weights/best.pt exists: True
STABLE YOLO results.csv: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab/results.csv exists: True
STABLE YOLO epochs: 30


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
25,26,390.302,1.87272,1.60086,1.25133,0.50000,0.56552,0.49535,0.20935,1.91642,1.58180,1.30507,0.000350,0.000350,0.000350
26,27,403.436,1.85931,1.63125,1.23196,0.47561,0.55045,0.50841,0.22807,1.87148,1.55491,1.27199,0.000284,0.000284,0.000284
27,28,416.729,1.84009,1.63754,1.22470,0.51329,0.53821,0.51853,0.23432,1.83300,1.50185,1.25394,0.000218,0.000218,0.000218
28,29,429.837,1.81727,1.59613,1.23711,0.53052,0.57241,0.55470,0.24955,1.82812,1.55066,1.24967,0.000152,0.000152,0.000152
29,30,443.319,1.79738,1.55943,1.22804,0.51046,0.55374,0.53555,0.24151,1.83378,1.52233,1.24812,0.000086,0.000086,0.000086



LATEST YOLO best weights: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab-9/weights/best.pt exists: True
LATEST YOLO results.csv: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab-9/results.csv exists: True
LATEST YOLO epochs: 30


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
25,26,781.510,1.80355,1.51911,1.18941,0.66139,0.47222,0.49310,0.22711,1.84018,1.52453,1.20167,0.000350,0.000350,0.000350
26,27,809.200,1.82862,1.54068,1.19615,0.69699,0.46667,0.51178,0.24265,1.83509,1.46287,1.20429,0.000284,0.000284,0.000284
27,28,836.500,1.80620,1.56530,1.18937,0.70929,0.48611,0.52606,0.24082,1.79449,1.48883,1.18187,0.000218,0.000218,0.000218
28,29,863.721,1.78337,1.50190,1.19581,0.67149,0.53611,0.54813,0.26427,1.76836,1.43764,1.17300,0.000152,0.000152,0.000152
29,30,891.019,1.80514,1.50904,1.19259,0.65779,0.50278,0.53727,0.26214,1.73799,1.43258,1.16260,0.000086,0.000086,0.000086


In [ ]:
# 10) Fine-tune RT-DETR-L
# RT-DETR is heavier than YOLO. This cell now streams progress live.
# Safety guard in Cell 1 prevents accidentally running RT-DETR-L on slow GPUs like T4.
if RUN_RTDETR_TRAINING:
    cmd = [
        sys.executable,
        "-u",
        "code/code/scripts/train/train_rtdetr.py",
        "--data", str(DATA_YAML),
        "--weights", "rtdetr-l.pt",
        "--epochs", str(EPOCHS_RTDETR),
        "--imgsz", str(IMGSZ),
        "--batch", str(BATCH_RTDETR),
        "--device", "0",
        "--name", RTDETR_RUN_NAME,
        "--workers", "2",
    ]
    run_command(cmd)
else:
    print("Skipping RT-DETR training.")


Streaming output truncated to the last 5000 lines.
        2/5      5.71G      1.185     0.4961     0.1642          3       1280: 33% ━━━━──────── 424/1260 3.5it/s 2:05<4:02
        2/5      5.71G      1.184     0.4965      0.164          6       1280: 33% ━━━━──────── 425/1260 3.5it/s 2:05<4:01
        2/5      5.71G      1.183     0.4965      0.164          1       1280: 33% ━━━━──────── 426/1260 3.5it/s 2:05<3:60
        2/5      5.71G      1.183     0.4961     0.1642          7       1280: 33% ━━━━──────── 427/1260 3.5it/s 2:06<4:01
        2/5      5.71G      1.183     0.4959      0.164          4       1280: 33% ━━━━──────── 428/1260 3.5it/s 2:06<4:00
        2/5      5.71G      1.187     0.4957     0.1653          4       1280: 34% ━━━━──────── 429/1260 3.5it/s 2:06<3:60
        2/5      5.71G      1.186     0.4957     0.1651          6       1280: 34% ━━━━──────── 430/1260 3.3it/s 2:07<4:12
        2/5      5.71G      1.186     0.4957     0.1652          6       1280: 34% ━━━━─

In [ ]:
# 11) Sync latest Ultralytics runs to stable folders expected by eval config
# This fixes the common Colab/Ultralytics issue where new training runs are saved as -2, -3, ...
# while the eval config still points to the original stable folder name.

if RUN_YOLO_TRAINING or RUN_YOLO_INFERENCE:
    sync_latest_run_to_stable(
        PROJECT_ROOT / "outputs/training/yolo",
        f"{YOLO_RUN_NAME}*",
        YOLO_RUN_NAME,
    )

if RUN_RTDETR_TRAINING or RUN_RTDETR_INFERENCE:
    sync_latest_run_to_stable(
        PROJECT_ROOT / "outputs/training/rtdetr",
        f"{RTDETR_RUN_NAME}*",
        RTDETR_RUN_NAME,
    )

# Show final stable folders and epoch counts
for label, root, run_name in [
    ("YOLO", PROJECT_ROOT / "outputs/training/yolo", YOLO_RUN_NAME),
    ("RT-DETR", PROJECT_ROOT / "outputs/training/rtdetr", RTDETR_RUN_NAME),
]:
    stable = root / run_name
    results_csv = stable / "results.csv"
    weights = stable / "weights/best.pt"
    print(f"\n{label} stable folder: {stable}")
    print("  weights exists:", weights.exists())
    print("  results exists:", results_csv.exists())
    if results_csv.exists():
        df = pd.read_csv(results_csv)
        print("  epochs:", len(df))
        display(df.tail())

Stable folder is already latest: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab
Stable folder is already latest: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/rtdetr/rtdetr_l_drone_colab

YOLO stable folder: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab
  weights exists: True
  results exists: True
  epochs: 30


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
25,26,781.510,1.80355,1.51911,1.18941,0.66139,0.47222,0.49310,0.22711,1.84018,1.52453,1.20167,0.000350,0.000350,0.000350
26,27,809.200,1.82862,1.54068,1.19615,0.69699,0.46667,0.51178,0.24265,1.83509,1.46287,1.20429,0.000284,0.000284,0.000284
27,28,836.500,1.80620,1.56530,1.18937,0.70929,0.48611,0.52606,0.24082,1.79449,1.48883,1.18187,0.000218,0.000218,0.000218
28,29,863.721,1.78337,1.50190,1.19581,0.67149,0.53611,0.54813,0.26427,1.76836,1.43764,1.17300,0.000152,0.000152,0.000152
29,30,891.019,1.80514,1.50904,1.19259,0.65779,0.50278,0.53727,0.26214,1.73799,1.43258,1.16260,0.000086,0.000086,0.000086



RT-DETR stable folder: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/rtdetr/rtdetr_l_drone_colab
  weights exists: True
  results exists: True
  epochs: 5


,epoch,time,train/giou_loss,train/cls_loss,train/l1_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/giou_loss,val/cls_loss,val/l1_loss,lr/pg0,lr/pg1,lr/pg2
0,1,395.892,1.68418,0.87376,0.38718,0.55050,0.36389,0.36501,0.11946,1.43877,0.43559,0.32136,0.000666,0.000666,0.000666
1,2,776.801,1.25774,0.49599,0.18342,0.62179,0.38333,0.40676,0.16939,1.13768,0.65327,0.19323,0.001069,0.001069,0.001069
2,3,1157.100,1.23123,0.47816,0.17890,0.50781,0.33333,0.32152,0.09621,NaN,NaN,NaN,0.001208,0.001208,0.001208
3,4,1537.110,1.26169,0.53930,0.19253,0.40188,0.33611,0.28951,0.11797,NaN,NaN,NaN,0.000812,0.000812,0.000812
4,5,1918.650,1.19350,0.50593,0.17406,0.55561,0.44453,0.42175,0.17061,NaN,NaN,NaN,0.000416,0.000416,0.000416


In [ ]:
# 12) Inference on each held-out evaluation subset
# Runs each enabled detector on:
# - test_drone_positive
# - test_hard_negative
# - test_mixed

models_to_run = []
if RUN_GDINO:
    models_to_run.append("grounding_dino")
if RUN_YOLO_INFERENCE:
    models_to_run.append("yolo11n_drone_finetuned")
if RUN_RTDETR_INFERENCE:
    models_to_run.append("rtdetr_l_drone_finetuned")

for eval_name, job in EVAL_JOBS.items():
    print("\n" + "#" * 100)
    print("EVAL SUBSET:", eval_name, "| source subset:", job["subset"], "| images:", job["n_images"])
    print("#" * 100)

    # Delete stale predictions for this eval subset.
    pred_dir = job["output_root"] / "predictions"
    pred_dir.mkdir(parents=True, exist_ok=True)
    if RESET_PREDICTIONS:
        for model_name in models_to_run:
            p = pred_dir / f"{model_name}_predictions.csv"
            if p.exists():
                p.unlink()
                print("Deleted stale prediction file:", p)

    base_cmd = [
        sys.executable,
        "code/code/scripts/eval/run_detectors.py",
        "--config", str(job["config"]),
        "--project-root", str(PROJECT_ROOT),
    ]

    if RUN_GDINO:
        run_command(base_cmd + ["--models", "grounding_dino"], check=True)

    if RUN_YOLO_INFERENCE:
        run_command(base_cmd + ["--models", "yolo11n_drone_finetuned"], check=False)

    if RUN_RTDETR_INFERENCE:
        run_command(base_cmd + ["--models", "rtdetr_l_drone_finetuned"], check=False)



####################################################################################################
EVAL SUBSET: test_drone_positive | source subset: synthetic_drone_positive | images: 180
####################################################################################################
Deleted stale prediction file: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_drone_positive/predictions/grounding_dino_predictions.csv
Deleted stale prediction file: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_drone_positive/predictions/yolo11n_drone_finetuned_predictions.csv
Deleted stale prediction file: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_drone_positive/predictions/rtdetr_l_drone_finetuned_predictions.csv

Running:
/usr/bin/p

In [ ]:
# 13) Inspect prediction CSVs by evaluation subset
import pandas as pd

for eval_name, job in EVAL_JOBS.items():
    pred_dir = job["output_root"] / "predictions"
    print("\n" + "#" * 100)
    print("EVAL SUBSET:", eval_name)
    print("Prediction directory:", pred_dir)
    print("#" * 100)

    if not pred_dir.exists():
        print("Prediction directory does not exist.")
        continue

    for csv_path in sorted(pred_dir.glob("*_predictions.csv")):
        df = pd.read_csv(csv_path)
        print("\n" + "=" * 80)
        print(csv_path.name)
        print("shape:", df.shape)
        if "model_name" in df.columns and len(df):
            print("model_name:", df["model_name"].unique())
        if "confidence" in df.columns and len(df):
            print("confidence min/mean/max:", df["confidence"].min(), df["confidence"].mean(), df["confidence"].max())
        if "image_id" in df.columns and len(df):
            print("first image_ids:", df["image_id"].head().tolist())
        display(df.head())



####################################################################################################
EVAL SUBSET: test_drone_positive
Prediction directory: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_drone_positive/predictions
####################################################################################################

grounding_dino_predictions.csv
shape: (2061, 14)
model_name: ['grounding_dino']
confidence min/mean/max: 0.200184 0.35945766424065984 0.9426
first image_ids: ['img_000031.png', 'img_000031.png', 'img_000031.png', 'img_000031.png', 'img_000031.png']


,image_id,image_path,model_name,pred_class_id,pred_class_name,confidence,x1,y1,x2,y2,width,height,prompt,raw_label
0,img_000031.png,/content/drive/MyDrive/Synthetic Stress Testin...,grounding_dino,0,drone,0.875487,212.90,193.82,244.68,212.37,31.78,18.55,drone,drone
1,img_000031.png,/content/drive/MyDrive/Synthetic Stress Testin...,grounding_dino,0,drone,0.897025,212.87,193.87,244.73,212.44,31.87,18.57,quadcopter,quadcopter
2,img_000031.png,/content/drive/MyDrive/Synthetic Stress Testin...,grounding_dino,0,drone,0.826014,212.48,193.10,245.17,212.92,32.69,19.82,unmanned aerial vehicle,unmanned aerial vehicle
3,img_000031.png,/content/drive/MyDrive/Synthetic Stress Testin...,grounding_dino,0,drone,0.614465,211.35,190.64,246.53,214.52,35.18,23.88,small drone in the sky,small drone
4,img_000031.png,/content/drive/MyDrive/Synthetic Stress Testin...,grounding_dino,0,drone,0.473155,1.74,1.40,638.24,341.38,636.50,339.98,small drone in the sky,sky



rtdetr_l_drone_finetuned_predictions.csv
shape: (5862, 14)
model_name: ['rtdetr_l_drone_finetuned']
confidence min/mean/max: 0.050003 0.09204810252473558 0.790581
first image_ids: ['img_000031.png', 'img_000031.png', 'img_000031.png', 'img_000031.png', 'img_000031.png']


,image_id,image_path,model_name,pred_class_id,pred_class_name,confidence,x1,y1,x2,y2,width,height,prompt,raw_label
0,img_000031.png,/content/drive/MyDrive/Synthetic Stress Testin...,rtdetr_l_drone_finetuned,0,drone,0.720437,208.16,189.84,248.97,217.50,40.81,27.66,NaN,drone
1,img_000031.png,/content/drive/MyDrive/Synthetic Stress Testin...,rtdetr_l_drone_finetuned,0,drone,0.178034,621.37,264.62,638.23,280.88,16.86,16.26,NaN,drone
2,img_000031.png,/content/drive/MyDrive/Synthetic Stress Testin...,rtdetr_l_drone_finetuned,0,drone,0.137040,623.49,266.63,637.95,278.52,14.46,11.90,NaN,drone
3,img_000031.png,/content/drive/MyDrive/Synthetic Stress Testin...,rtdetr_l_drone_finetuned,0,drone,0.115599,624.51,321.42,637.64,334.95,13.13,13.52,NaN,drone
4,img_000031.png,/content/drive/MyDrive/Synthetic Stress Testin...,rtdetr_l_drone_finetuned,0,drone,0.101073,507.00,613.21,526.82,627.98,19.81,14.78,NaN,drone



yolo11n_drone_finetuned_predictions.csv
shape: (313, 14)
model_name: ['yolo11n_drone_finetuned']
confidence min/mean/max: 0.050284 0.2728070127795527 0.779578
first image_ids: ['img_000031.png', 'img_000031.png', 'img_000077.png', 'img_000077.png', 'img_000077.png']


...

,image_id,image_path,model_name,pred_class_id,pred_class_name,confidence,x1,y1,x2,y2,width,height,prompt,raw_label
0,img_000020.png,/content/drive/MyDrive/Synthetic Stress Testin...,grounding_dino,0,drone,0.703800,244.20,177.67,274.82,199.48,30.63,21.82,drone,drone
1,img_000020.png,/content/drive/MyDrive/Synthetic Stress Testin...,grounding_dino,0,drone,0.671314,50.46,256.54,69.35,285.16,18.89,28.62,drone,drone
2,img_000020.png,/content/drive/MyDrive/Synthetic Stress Testin...,grounding_dino,0,drone,0.661020,243.95,177.77,274.87,199.63,30.91,21.85,quadcopter,quadcopter
3,img_000020.png,/content/drive/MyDrive/Synthetic Stress Testin...,grounding_dino,0,drone,0.639239,50.30,256.49,69.43,285.25,19.13,28.76,quadcopter,quadcopter
4,img_000020.png,/content/drive/MyDrive/Synthetic Stress Testin...,grounding_dino,0,drone,0.463423,243.59,176.94,275.21,200.72,31.62,23.78,unmanned aerial vehicle,unmanned aerial vehicle



rtdetr_l_drone_finetuned_predictions.csv
shape: (6401, 14)
model_name: ['rtdetr_l_drone_finetuned']
confidence min/mean/max: 0.05 0.09401114888298705 0.781238
first image_ids: ['img_000020.png', 'img_000020.png', 'img_000020.png', 'img_000020.png', 'img_000020.png']


,image_id,image_path,model_name,pred_class_id,pred_class_name,confidence,x1,y1,x2,y2,width,height,prompt,raw_label
0,img_000020.png,/content/drive/MyDrive/Synthetic Stress Testin...,rtdetr_l_drone_finetuned,0,drone,0.758449,237.98,173.56,282.19,205.20,44.21,31.64,NaN,drone
1,img_000020.png,/content/drive/MyDrive/Synthetic Stress Testin...,rtdetr_l_drone_finetuned,0,drone,0.272862,578.79,421.19,586.95,426.60,8.15,5.41,NaN,drone
2,img_000020.png,/content/drive/MyDrive/Synthetic Stress Testin...,rtdetr_l_drone_finetuned,0,drone,0.258142,234.55,441.82,241.54,446.37,6.99,4.56,NaN,drone
3,img_000020.png,/content/drive/MyDrive/Synthetic Stress Testin...,rtdetr_l_drone_finetuned,0,drone,0.182025,454.67,421.11,460.59,425.30,5.92,4.20,NaN,drone
4,img_000020.png,/content/drive/MyDrive/Synthetic Stress Testin...,rtdetr_l_drone_finetuned,0,drone,0.167585,483.76,439.62,505.15,458.64,21.38,19.02,NaN,drone



yolo11n_drone_finetuned_predictions.csv
shape: (382, 14)
model_name: ['yolo11n_drone_finetuned']
confidence min/mean/max: 0.050366 0.2563887565445026 0.770713
first image_ids: ['img_000020.png', 'img_000020.png', 'img_000020.png', 'img_000047.png', 'img_000047.png']


,image_id,image_path,model_name,pred_class_id,pred_class_name,confidence,x1,y1,x2,y2,width,height,prompt,raw_label
0,img_000020.png,/content/drive/MyDrive/Synthetic Stress Testin...,yolo11n_drone_finetuned,0,drone,0.683181,234.66,174.02,288.34,206.18,53.68,32.16,NaN,drone
1,img_000020.png,/content/drive/MyDrive/Synthetic Stress Testin...,yolo11n_drone_finetuned,0,drone,0.288453,231.43,171.16,294.19,210.67,62.77,39.51,NaN,drone
2,img_000020.png,/content/drive/MyDrive/Synthetic Stress Testin...,yolo11n_drone_finetuned,0,drone,0.060236,238.93,177.33,282.34,202.61,43.42,25.28,NaN,drone
3,img_000047.png,/content/drive/MyDrive/Synthetic Stress Testin...,yolo11n_drone_finetuned,0,drone,0.377058,30.58,186.77,50.42,199.87,19.84,13.09,NaN,drone
4,img_000047.png,/content/drive/MyDrive/Synthetic Stress Testin...,yolo11n_drone_finetuned,0,drone,0.306000,367.19,110.77,375.87,116.25,8.68,5.48,NaN,drone


In [ ]:
# #Fix the evaluator
# from pathlib import Path

# metrics_py = PROJECT_ROOT / "src/drone_stress/detector_eval/metrics.py"
# print("Patching:", metrics_py)
# text = metrics_py.read_text()

# old = """old = pd.read_csv(win_long_path(path))"""

# new = """try:
#             old = pd.read_csv(win_long_path(path))
#         except pd.errors.EmptyDataError:
#             old = pd.DataFrame()
#         except Exception as e:
#             print(f"WARNING: failed to read existing metrics CSV {path}: {e}")
#             old = pd.DataFrame()"""

# if old not in text:
#     print("Exact target line not found. Showing nearby lines containing pd.read_csv:")
#     for i, line in enumerate(text.splitlines(), start=1):
#         if "pd.read_csv" in line:
#             print(i, line)
# else:
#     text = text.replace(old, new)
#     metrics_py.write_text(text)
#     print("✅ Patched metrics.py to tolerate empty/corrupt existing CSV files.")

Patching: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/src/drone_stress/detector_eval/metrics.py
✅ Patched metrics.py to tolerate empty/corrupt existing CSV files.


In [ ]:
# #clean the old broken metrics files
# from pathlib import Path

# for eval_name, job in EVAL_JOBS.items():
#     metrics_dir = job["output_root"] / "metrics"
#     print("\nCleaning:", metrics_dir)
#     if metrics_dir.exists():
#         for p in metrics_dir.glob("*.csv"):
#             print(" deleting", p.name)
#             p.unlink()

# combined_metrics_dir = EVAL_OUTPUT_ROOT / "combined_metrics"
# if combined_metrics_dir.exists():
#     for p in combined_metrics_dir.glob("*.csv"):
#         print(" deleting combined", p.name)
#         p.unlink()


Cleaning: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_drone_positive/metrics
 deleting grounding_dino_matched_predictions.csv
 deleting grounding_dino_image_level_metrics.csv
 deleting grounding_dino_summary_metrics.csv
 deleting grounding_dino_grouped_metrics.csv
 deleting recall_by_subset.csv
 deleting recall_by_size_noise_blur.csv
 deleting false_positive_by_distractor_type.csv
 deleting recall_by_distractor_present.csv
 deleting image_level_metrics.csv
 deleting matched_predictions.csv
 deleting yolo11n_drone_finetuned_matched_predictions.csv
 deleting yolo11n_drone_finetuned_image_level_metrics.csv
 deleting yolo11n_drone_finetuned_summary_metrics.csv
 deleting yolo11n_drone_finetuned_grouped_metrics.csv
 deleting rtdetr_l_drone_finetuned_matched_predictions.csv
 deleting rtdetr_l_drone_finetuned_image_level_metrics.csv
 deleting rtdetr_l_drone_finetuned_summary_metrics.csv
 deleting rtdetr_

In [ ]:
#fix metrics.py identation
from pathlib import Path
import re
import py_compile

metrics_py = PROJECT_ROOT / "src/drone_stress/detector_eval/metrics.py"
print("Repairing:", metrics_py)

text = metrics_py.read_text()

# Fix the malformed try/except block around:
# old = pd.read_csv(win_long_path(path))
pattern = re.compile(
    r'(?m)^(?P<indent>\s*)try:\n'
    r'\s*old = pd\.read_csv\(win_long_path\(path\)\)\n'
    r'\s*except pd\.errors\.EmptyDataError:\n'
    r'\s*old = pd\.DataFrame\(\)\n'
    r'\s*except Exception as e:\n'
    r'\s*print\(f"WARNING: failed to read existing metrics CSV \{path\}: \{e\}"\)\n'
    r'\s*old = pd\.DataFrame\(\)'
)

def repl(m):
    indent = m.group("indent")
    return (
        f"{indent}try:\n"
        f"{indent}    old = pd.read_csv(win_long_path(path))\n"
        f"{indent}except pd.errors.EmptyDataError:\n"
        f"{indent}    old = pd.DataFrame()\n"
        f"{indent}except Exception as e:\n"
        f"{indent}    print(f\"WARNING: failed to read existing metrics CSV {{path}}: {{e}}\")\n"
        f"{indent}    old = pd.DataFrame()"
    )

new_text, n = pattern.subn(repl, text)

if n == 0:
    print("Regex did not find the malformed block. Showing lines 330-350:")
    lines = text.splitlines()
    for i in range(329, min(350, len(lines))):
        print(f"{i+1}: {lines[i]}")
    raise RuntimeError("Patch target not found. Paste the printed lines here.")

metrics_py.write_text(new_text)

# Verify syntax
py_compile.compile(str(metrics_py), doraise=True)

print("✅ metrics.py syntax repaired.")

Repairing: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/src/drone_stress/detector_eval/metrics.py
✅ metrics.py syntax repaired.


In [ ]:
#Clean broken metrics
for eval_name, job in EVAL_JOBS.items():
    metrics_dir = job["output_root"] / "metrics"
    print("\nCleaning:", metrics_dir)
    if metrics_dir.exists():
        for p in metrics_dir.glob("*.csv"):
            print(" deleting", p.name)
            p.unlink()

combined_metrics_dir = EVAL_OUTPUT_ROOT / "combined_metrics"
if combined_metrics_dir.exists():
    for p in combined_metrics_dir.glob("*.csv"):
        print(" deleting combined", p.name)
        p.unlink()


Cleaning: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_drone_positive/metrics

Cleaning: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_hard_negative/metrics

Cleaning: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_mixed/metrics


In [ ]:
# 14) Metrics + prediction QA by held-out evaluation subset
# Writes one combined table with eval_subset, model_name, recall, FP rates/counts, etc.

import pandas as pd
from pathlib import Path

models = [
    "grounding_dino",
    "yolo11n_drone_finetuned",
    "rtdetr_l_drone_finetuned",
]

all_rows = []

for eval_name, job in EVAL_JOBS.items():
    print("\n" + "#" * 100)
    print("METRICS FOR EVAL SUBSET:", eval_name)
    print("#" * 100)

    metadata_df = pd.read_csv(job["dataset_root"] / "metadata.csv")
    n_target_present = int(metadata_df["target_present"].astype(bool).sum()) if "target_present" in metadata_df.columns else 0
    n_distractor_only = int((metadata_df["subset"] == "synthetic_distractor_only").sum()) if "subset" in metadata_df.columns else 0

    pred_dir = job["output_root"] / "predictions"
    metrics_dir = job["output_root"] / "metrics"
    metrics_dir.mkdir(parents=True, exist_ok=True)

    subset_rows = []

    for model in models:
        pred_csv = pred_dir / f"{model}_predictions.csv"
        print("\n" + "=" * 80)
        print("MODEL:", model)
        print("=" * 80)

        if not pred_csv.is_file():
            print("SKIP missing prediction file:", pred_csv)
            continue

        pred_df = pd.read_csv(pred_csv)
        print("prediction rows:", len(pred_df))

        if len(pred_df) == 0:
            print("Empty prediction CSV. Adding zero-detection metrics instead of crashing.")
            for iou_thr in [0.25, 0.50]:
                subset_rows.append({
                    "eval_subset": eval_name,
                    "source_subset": job["subset"],
                    "model_name": model,
                    "model_type": "empty_or_underconfident",
                    "iou_threshold": iou_thr,
                    "n_target_present": n_target_present,
                    "n_distractor_only": n_distractor_only,
                    "recall": 0.0,
                    "tp": 0,
                    "fn": n_target_present,
                    "mean_best_iou": 0.0,
                    "false_positive_image_rate": 0.0,
                    "fp_count": 0,
                    "note": "No predictions at configured inference threshold; likely undertrained or threshold too high.",
                })
            continue

        # Evaluate this model for this subset.
        result = run_command([
            sys.executable,
            "code/code/scripts/eval/evaluate_detector_predictions.py",
            "--config", str(job["config"]),
            "--predictions", str(pred_csv),
            "--project-root", str(PROJECT_ROOT),
        ], check=False)

        #if result.returncode == 0:
        rc = result if isinstance(result, int) else result.returncode

        if rc == 0:
            summary_path = metrics_dir / "summary_by_model.csv"
            if summary_path.exists():
                summary_df = pd.read_csv(summary_path)
                summary_df["eval_subset"] = eval_name
                summary_df["source_subset"] = job["subset"]
                subset_rows.extend(summary_df.to_dict("records"))
        else:
            print("Evaluation failed for this model; see output above.")
            continue

        # Visualization / prediction QA
        run_command([
            sys.executable,
            "code/code/scripts/eval/visualize_predictions.py",
            "--config", str(job["config"]),
            "--predictions", str(pred_csv),
            "--project-root", str(PROJECT_ROOT),
            "--num-samples", "12",
        ], check=False)

    subset_summary = pd.DataFrame(subset_rows)
    if len(subset_summary):
        subset_summary = subset_summary.drop_duplicates(
            subset=["eval_subset", "model_name", "iou_threshold"],
            keep="last",
        ).sort_values(["eval_subset", "model_name", "iou_threshold"]).reset_index(drop=True)

    subset_summary_path = metrics_dir / "summary_by_model_with_eval_subset.csv"
    subset_summary.to_csv(subset_summary_path, index=False)
    print("\nSubset summary written to:", subset_summary_path)
    display(subset_summary)

    all_rows.extend(subset_summary.to_dict("records"))

combined_summary = pd.DataFrame(all_rows)

if len(combined_summary):
    combined_summary = combined_summary.drop_duplicates(
        subset=["eval_subset", "model_name", "iou_threshold"],
        keep="last",
    ).sort_values(["eval_subset", "model_name", "iou_threshold"]).reset_index(drop=True)

combined_metrics_dir = EVAL_OUTPUT_ROOT / "combined_metrics"
combined_metrics_dir.mkdir(parents=True, exist_ok=True)

combined_path = combined_metrics_dir / "summary_by_model_by_eval_subset.csv"
combined_summary.to_csv(combined_path, index=False)

print("\nCombined final-clean summary written to:")
print(combined_path)
display(combined_summary)

# Optional pivots for quick comparison.
if len(combined_summary):
    print("\nRecall pivot:")
    display(combined_summary.pivot_table(
        index=["model_name", "iou_threshold"],
        columns="eval_subset",
        values="recall",
        aggfunc="first",
    ))

    print("\nFP count pivot:")
    display(combined_summary.pivot_table(
        index=["model_name", "iou_threshold"],
        columns="eval_subset",
        values="fp_count",
        aggfunc="first",
    ))



####################################################################################################
METRICS FOR EVAL SUBSET: test_drone_positive
####################################################################################################

MODEL: grounding_dino
prediction rows: 2061

Running:
/usr/bin/python3 scripts/eval/evaluate_detector_predictions.py --config '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs/generated_eval_subsets/eval_test_drone_positive.yaml' --predictions '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_drone_positive/predictions/grounding_dino_predictions.csv' --project-root '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification'

----- live output starts -----
Evaluating grounding_dino from /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_cle

,model_name,model_type,iou_threshold,n_target_present,n_distractor_only,recall,tp,fn,mean_best_iou,false_positive_image_rate,fp_count,note,eval_subset,source_subset
0,grounding_dino,grounding_dino,0.25,180,0,0.622222,112,68,0.399236,NaN,0,Open-vocabulary drone prompts; boxes count as ...,test_drone_positive,synthetic_drone_positive
1,grounding_dino,grounding_dino,0.50,180,0,0.538889,97,83,0.366148,NaN,0,Open-vocabulary drone prompts; boxes count as ...,test_drone_positive,synthetic_drone_positive
2,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.25,180,0,0.816667,147,33,0.547084,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive
3,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.50,180,0,0.700000,126,54,0.502650,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive
4,yolo11n_drone_finetuned,ultralytics_yolo,0.25,143,0,0.916084,131,12,0.677089,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive
5,yolo11n_drone_finetuned,ultralytics_yolo,0.50,143,0,0.860140,123,20,0.654746,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive



####################################################################################################
METRICS FOR EVAL SUBSET: test_hard_negative
####################################################################################################

MODEL: grounding_dino
prediction rows: 1873

Running:
/usr/bin/python3 scripts/eval/evaluate_detector_predictions.py --config '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs/generated_eval_subsets/eval_test_hard_negative.yaml' --predictions '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_hard_negative/predictions/grounding_dino_predictions.csv' --project-root '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification'

----- live output starts -----
Evaluating grounding_dino from /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_

,model_name,model_type,iou_threshold,n_target_present,n_distractor_only,recall,tp,fn,mean_best_iou,false_positive_image_rate,fp_count,note,eval_subset,source_subset
0,grounding_dino,grounding_dino,0.25,0,180,NaN,0,0,NaN,1.0,1873,Open-vocabulary drone prompts; boxes count as ...,test_hard_negative,synthetic_distractor_only
1,grounding_dino,grounding_dino,0.50,0,180,NaN,0,0,NaN,1.0,1873,Open-vocabulary drone prompts; boxes count as ...,test_hard_negative,synthetic_distractor_only
2,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.25,0,180,NaN,0,0,NaN,1.0,7218,Fine-tuned / drone-capable checkpoint; drone_l...,test_hard_negative,synthetic_distractor_only
3,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.50,0,180,NaN,0,0,NaN,1.0,7218,Fine-tuned / drone-capable checkpoint; drone_l...,test_hard_negative,synthetic_distractor_only
4,yolo11n_drone_finetuned,ultralytics_yolo,0.25,0,62,NaN,0,0,NaN,1.0,107,Fine-tuned / drone-capable checkpoint; drone_l...,test_hard_negative,synthetic_distractor_only
5,yolo11n_drone_finetuned,ultralytics_yolo,0.50,0,62,NaN,0,0,NaN,1.0,107,Fine-tuned / drone-capable checkpoint; drone_l...,test_hard_negative,synthetic_distractor_only



####################################################################################################
METRICS FOR EVAL SUBSET: test_mixed
####################################################################################################

MODEL: grounding_dino
prediction rows: 1999

Running:
/usr/bin/python3 scripts/eval/evaluate_detector_predictions.py --config '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs/generated_eval_subsets/eval_test_mixed.yaml' --predictions '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_mixed/predictions/grounding_dino_predictions.csv' --project-root '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification'

----- live output starts -----
Evaluating grounding_dino from /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_mix

...

,model_name,model_type,iou_threshold,n_target_present,n_distractor_only,recall,tp,fn,mean_best_iou,false_positive_image_rate,fp_count,note,eval_subset,source_subset
0,grounding_dino,grounding_dino,0.25,180,0,0.622222,112,68,0.399236,NaN,0,Open-vocabulary drone prompts; boxes count as ...,test_drone_positive,synthetic_drone_positive
1,grounding_dino,grounding_dino,0.50,180,0,0.538889,97,83,0.366148,NaN,0,Open-vocabulary drone prompts; boxes count as ...,test_drone_positive,synthetic_drone_positive
2,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.25,180,0,0.816667,147,33,0.547084,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive
3,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.50,180,0,0.700000,126,54,0.502650,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive
4,yolo11n_drone_finetuned,ultralytics_yolo,0.25,143,0,0.916084,131,12,0.677089,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive
5,yolo11n_drone_finetuned,ultralytics_yolo,0.50,143,0,0.860140,123,20,0.654746,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive
6,grounding_dino,grounding_dino,0.25,0,180,NaN,0,0,NaN,1.0,1873,Open-vocabulary drone prompts; boxes count as ...,test_hard_negative,synthetic_distractor_only
7,grounding_dino,grounding_dino,0.50,0,180,NaN,0,0,NaN,1.0,1873,Open-vocabulary drone prompts; boxes count as ...,test_hard_negative,synthetic_distractor_only
8,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.25,0,180,NaN,0,0,NaN,1.0,7218,Fine-tuned / drone-capable checkpoint; drone_l...,test_hard_negative,synthetic_distractor_only
9,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.50,0,180,NaN,0,0,NaN,1.0,7218,Fine-tuned / drone-capable checkpoint; drone_l...,test_hard_negative,synthetic_distractor_only



Recall pivot:


eval_subset                             test_drone_positive  test_mixed
model_name               iou_threshold                                 
grounding_dino           0.25                      0.622222    0.450000
                         0.50                      0.538889    0.372222
rtdetr_l_drone_finetuned 0.25                      0.816667    0.827778
                         0.50                      0.700000    0.694444
yolo11n_drone_finetuned  0.25                      0.916084    0.828025
                         0.50                      0.860140    0.777070


FP count pivot:


eval_subset                             test_drone_positive  \
model_name               iou_threshold                        
grounding_dino           0.25                             0   
                         0.50                             0   
rtdetr_l_drone_finetuned 0.25                             0   
                         0.50                             0   
yolo11n_drone_finetuned  0.25                             0   
                         0.50                             0   

eval_subset                             test_hard_negative  test_mixed  
model_name               iou_threshold                                  
grounding_dino           0.25                         1873           0  
                         0.50                         1873           0  
rtdetr_l_drone_finetuned 0.25                         7218           0  
                         0.50                         7218           0  
yolo11n_drone_finetuned  0.25                          107           0  
                         0.50                          107           0

In [ ]:
#quick diagnostic for YOLO evaluator reports
import pandas as pd
from pathlib import Path

for eval_name, job in EVAL_JOBS.items():
    print("\n", eval_name)
    meta = pd.read_csv(job["dataset_root"] / "metadata.csv")
    print("metadata rows:", len(meta))
    print("metadata target_present:", int(meta["target_present"].astype(bool).sum()))
    print("metadata subset counts:")
    print(meta["subset"].value_counts())

    pred_dir = job["output_root"] / "predictions"
    for model in ["grounding_dino", "yolo11n_drone_finetuned", "rtdetr_l_drone_finetuned"]:
        pred_csv = pred_dir / f"{model}_predictions.csv"
        if pred_csv.exists():
            pred = pd.read_csv(pred_csv)
            if "image_id" in pred.columns:
                print(model, "prediction rows:", len(pred), "unique images:", pred["image_id"].nunique())
            elif "filename" in pred.columns:
                print(model, "prediction rows:", len(pred), "unique filenames:", pred["filename"].nunique())
            else:
                print(model, "prediction rows:", len(pred), "columns:", list(pred.columns))
        else:
            print(model, "missing")


 test_drone_positive
metadata rows: 180
metadata target_present: 180
metadata subset counts:
subset
synthetic_drone_positive    180
Name: count, dtype: int64
grounding_dino prediction rows: 2061 unique images: 180
yolo11n_drone_finetuned prediction rows: 313 unique images: 143
rtdetr_l_drone_finetuned prediction rows: 5862 unique images: 180

 test_hard_negative
metadata rows: 180
metadata target_present: 0
metadata subset counts:
subset
synthetic_distractor_only    180
Name: count, dtype: int64
grounding_dino prediction rows: 1873 unique images: 180
yolo11n_drone_finetuned prediction rows: 107 unique images: 62
rtdetr_l_drone_finetuned prediction rows: 7218 unique images: 180

 test_mixed
metadata rows: 180
metadata target_present: 180
metadata subset counts:
subset
synthetic_drone_plus_distractor    180
Name: count, dtype: int64
grounding_dino prediction rows: 1999 unique images: 180
yolo11n_drone_finetuned prediction rows: 382 unique images: 157
rtdetr_l_drone_finetuned prediction 

In [ ]:
#Create corrected final metrics table
import pandas as pd
import numpy as np
from pathlib import Path

combined_path = EVAL_OUTPUT_ROOT / "combined_metrics" / "summary_by_model_by_eval_subset.csv"
df = pd.read_csv(combined_path)

def pred_unique_images(pred_csv):
    if not Path(pred_csv).exists():
        return 0
    p = pd.read_csv(pred_csv)
    if len(p) == 0:
        return 0
    if "image_id" in p.columns:
        return p["image_id"].nunique()
    if "filename" in p.columns:
        return p["filename"].nunique()
    return 0

corrected_rows = []

for _, row in df.iterrows():
    row = row.copy()
    eval_name = row["eval_subset"]
    model = row["model_name"]
    job = EVAL_JOBS[eval_name]

    meta = pd.read_csv(job["dataset_root"] / "metadata.csv")
    total_images = len(meta)
    total_targets = int(meta["target_present"].astype(bool).sum())
    total_hardneg = int((meta["subset"] == "synthetic_distractor_only").sum())

    pred_csv = job["output_root"] / "predictions" / f"{model}_predictions.csv"
    unique_pred_images = pred_unique_images(pred_csv)

    row["metadata_total_images"] = total_images
    row["prediction_unique_images"] = unique_pred_images

    if total_targets > 0:
        # Positive or mixed subset.
        row["n_target_present"] = total_targets
        row["fn"] = total_targets - int(row["tp"])
        row["recall"] = int(row["tp"]) / total_targets
        row["false_positive_image_rate"] = np.nan
        row["fp_count"] = 0
    else:
        # Hard negative subset.
        row["n_target_present"] = 0
        row["n_distractor_only"] = total_hardneg
        row["recall"] = np.nan
        row["tp"] = 0
        row["fn"] = 0
        row["mean_best_iou"] = np.nan
        row["false_positive_image_rate"] = unique_pred_images / total_hardneg if total_hardneg else np.nan
        # fp_count from evaluator is still useful: number of false-positive boxes.
        row["fp_count"] = int(row["fp_count"])

    corrected_rows.append(row)

corrected = pd.DataFrame(corrected_rows)

corrected = corrected.sort_values(
    ["eval_subset", "model_name", "iou_threshold"]
).reset_index(drop=True)

corrected_path = EVAL_OUTPUT_ROOT / "combined_metrics" / "summary_by_model_by_eval_subset_CORRECTED.csv"
corrected.to_csv(corrected_path, index=False)

print("Corrected summary written to:")
print(corrected_path)

display(corrected)

print("\nCorrected recall pivot:")
display(corrected.pivot_table(
    index=["model_name", "iou_threshold"],
    columns="eval_subset",
    values="recall",
    aggfunc="first",
))

print("\nCorrected false-positive image-rate pivot:")
display(corrected.pivot_table(
    index=["model_name", "iou_threshold"],
    columns="eval_subset",
    values="false_positive_image_rate",
    aggfunc="first",
))

print("\nFP count pivot:")
display(corrected.pivot_table(
    index=["model_name", "iou_threshold"],
    columns="eval_subset",
    values="fp_count",
    aggfunc="first",
))

Corrected summary written to:
/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/combined_metrics/summary_by_model_by_eval_subset_CORRECTED.csv


,model_name,model_type,iou_threshold,n_target_present,n_distractor_only,recall,tp,fn,mean_best_iou,false_positive_image_rate,fp_count,note,eval_subset,source_subset,metadata_total_images,prediction_unique_images
0,grounding_dino,grounding_dino,0.25,180,0,0.622222,112,68,0.399236,NaN,0,Open-vocabulary drone prompts; boxes count as ...,test_drone_positive,synthetic_drone_positive,180,180
1,grounding_dino,grounding_dino,0.50,180,0,0.538889,97,83,0.366148,NaN,0,Open-vocabulary drone prompts; boxes count as ...,test_drone_positive,synthetic_drone_positive,180,180
2,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.25,180,0,0.816667,147,33,0.547084,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive,180,180
3,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.50,180,0,0.700000,126,54,0.502650,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive,180,180
4,yolo11n_drone_finetuned,ultralytics_yolo,0.25,180,0,0.727778,131,49,0.677089,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive,180,143
5,yolo11n_drone_finetuned,ultralytics_yolo,0.50,180,0,0.683333,123,57,0.654746,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive,180,143
6,grounding_dino,grounding_dino,0.25,0,180,NaN,0,0,NaN,1.000000,1873,Open-vocabulary drone prompts; boxes count as ...,test_hard_negative,synthetic_distractor_only,180,180
7,grounding_dino,grounding_dino,0.50,0,180,NaN,0,0,NaN,1.000000,1873,Open-vocabulary drone prompts; boxes count as ...,test_hard_negative,synthetic_distractor_only,180,180
8,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.25,0,180,NaN,0,0,NaN,1.000000,7218,Fine-tuned / drone-capable checkpoint; drone_l...,test_hard_negative,synthetic_distractor_only,180,180
9,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.50,0,180,NaN,0,0,NaN,1.000000,7218,Fine-tuned / drone-capable checkpoint; drone_l...,test_hard_negative,synthetic_distractor_only,180,180



Corrected recall pivot:


eval_subset                             test_drone_positive  test_mixed
model_name               iou_threshold                                 
grounding_dino           0.25                      0.622222    0.450000
                         0.50                      0.538889    0.372222
rtdetr_l_drone_finetuned 0.25                      0.816667    0.827778
                         0.50                      0.700000    0.694444
yolo11n_drone_finetuned  0.25                      0.727778    0.722222
                         0.50                      0.683333    0.677778


Corrected false-positive image-rate pivot:


eval_subset                             test_hard_negative
model_name               iou_threshold                    
grounding_dino           0.25                     1.000000
                         0.50                     1.000000
rtdetr_l_drone_finetuned 0.25                     1.000000
                         0.50                     1.000000
yolo11n_drone_finetuned  0.25                     0.344444
                         0.50                     0.344444


FP count pivot:


eval_subset                             test_drone_positive  \
model_name               iou_threshold                        
grounding_dino           0.25                             0   
                         0.50                             0   
rtdetr_l_drone_finetuned 0.25                             0   
                         0.50                             0   
yolo11n_drone_finetuned  0.25                             0   
                         0.50                             0   

eval_subset                             test_hard_negative  test_mixed  
model_name               iou_threshold                                  
grounding_dino           0.25                         1873           0  
                         0.50                         1873           0  
rtdetr_l_drone_finetuned 0.25                         7218           0  
                         0.50                         7218           0  
yolo11n_drone_finetuned  0.25                          107           0  
                         0.50                          107           0

In [ ]:
# 15) Correct final-clean metrics denominators
# Run this AFTER Cell 14.
# It creates:
# summary_by_model_by_eval_subset_CORRECTED.csv

import pandas as pd
import numpy as np
from pathlib import Path

combined_path = EVAL_OUTPUT_ROOT / "combined_metrics" / "summary_by_model_by_eval_subset.csv"
print("Reading original combined metrics:")
print(combined_path)
print("Exists:", combined_path.exists())

if not combined_path.exists():
    raise FileNotFoundError(
        "summary_by_model_by_eval_subset.csv does not exist. "
        "Run Cell 14 first."
    )

df = pd.read_csv(combined_path)

def get_unique_pred_images(pred_csv):
    pred_csv = Path(pred_csv)
    if not pred_csv.exists() or pred_csv.stat().st_size == 0:
        return 0

    pred = pd.read_csv(pred_csv)

    if len(pred) == 0:
        return 0

    if "image_id" in pred.columns:
        return pred["image_id"].nunique()

    if "filename" in pred.columns:
        return pred["filename"].nunique()

    return 0

corrected_rows = []

for _, row in df.iterrows():
    row = row.copy()

    eval_name = row["eval_subset"]
    model_name = row["model_name"]

    job = EVAL_JOBS[eval_name]

    metadata_path = job["dataset_root"] / "metadata.csv"
    metadata_df = pd.read_csv(metadata_path)

    total_images = len(metadata_df)
    total_target_present = int(metadata_df["target_present"].astype(bool).sum())

    if "subset" in metadata_df.columns:
        total_hard_negative = int((metadata_df["subset"] == "synthetic_distractor_only").sum())
    else:
        total_hard_negative = 0

    pred_csv = job["output_root"] / "predictions" / f"{model_name}_predictions.csv"
    unique_pred_images = get_unique_pred_images(pred_csv)

    row["metadata_total_images"] = total_images
    row["prediction_unique_images"] = unique_pred_images

    # Positive or mixed subset: recall denominator must be ALL target-present images,
    # not only images that had at least one prediction.
    if total_target_present > 0:
        tp = int(row["tp"])
        row["n_target_present"] = total_target_present
        row["fn"] = total_target_present - tp
        row["recall"] = tp / total_target_present
        row["false_positive_image_rate"] = np.nan
        row["fp_count"] = 0

    # Hard-negative subset: no recall. FP image rate denominator must be ALL hard-negative images.
    else:
        row["n_target_present"] = 0
        row["n_distractor_only"] = total_hard_negative
        row["tp"] = 0
        row["fn"] = 0
        row["recall"] = np.nan
        row["mean_best_iou"] = np.nan
        row["false_positive_image_rate"] = (
            unique_pred_images / total_hard_negative
            if total_hard_negative > 0
            else np.nan
        )
        row["fp_count"] = int(row["fp_count"])

    corrected_rows.append(row)

corrected = pd.DataFrame(corrected_rows)

corrected = corrected.sort_values(
    ["eval_subset", "model_name", "iou_threshold"]
).reset_index(drop=True)

corrected_path = EVAL_OUTPUT_ROOT / "combined_metrics" / "summary_by_model_by_eval_subset_CORRECTED.csv"
corrected.to_csv(corrected_path, index=False)

print("\n✅ Corrected summary written to:")
print(corrected_path)

display(corrected)

print("\nCorrected recall pivot:")
display(corrected.pivot_table(
    index=["model_name", "iou_threshold"],
    columns="eval_subset",
    values="recall",
    aggfunc="first",
))

print("\nCorrected false-positive image-rate pivot:")
display(corrected.pivot_table(
    index=["model_name", "iou_threshold"],
    columns="eval_subset",
    values="false_positive_image_rate",
    aggfunc="first",
))

print("\nFP count pivot:")
display(corrected.pivot_table(
    index=["model_name", "iou_threshold"],
    columns="eval_subset",
    values="fp_count",
    aggfunc="first",
))

Reading original combined metrics:
/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/combined_metrics/summary_by_model_by_eval_subset.csv
Exists: True

✅ Corrected summary written to:
/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/combined_metrics/summary_by_model_by_eval_subset_CORRECTED.csv


,model_name,model_type,iou_threshold,n_target_present,n_distractor_only,recall,tp,fn,mean_best_iou,false_positive_image_rate,fp_count,note,eval_subset,source_subset,metadata_total_images,prediction_unique_images
0,grounding_dino,grounding_dino,0.25,180,0,0.622222,112,68,0.399236,NaN,0,Open-vocabulary drone prompts; boxes count as ...,test_drone_positive,synthetic_drone_positive,180,180
1,grounding_dino,grounding_dino,0.50,180,0,0.538889,97,83,0.366148,NaN,0,Open-vocabulary drone prompts; boxes count as ...,test_drone_positive,synthetic_drone_positive,180,180
2,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.25,180,0,0.816667,147,33,0.547084,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive,180,180
3,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.50,180,0,0.700000,126,54,0.502650,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive,180,180
4,yolo11n_drone_finetuned,ultralytics_yolo,0.25,180,0,0.727778,131,49,0.677089,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive,180,143
5,yolo11n_drone_finetuned,ultralytics_yolo,0.50,180,0,0.683333,123,57,0.654746,NaN,0,Fine-tuned / drone-capable checkpoint; drone_l...,test_drone_positive,synthetic_drone_positive,180,143
6,grounding_dino,grounding_dino,0.25,0,180,NaN,0,0,NaN,1.000000,1873,Open-vocabulary drone prompts; boxes count as ...,test_hard_negative,synthetic_distractor_only,180,180
7,grounding_dino,grounding_dino,0.50,0,180,NaN,0,0,NaN,1.000000,1873,Open-vocabulary drone prompts; boxes count as ...,test_hard_negative,synthetic_distractor_only,180,180
8,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.25,0,180,NaN,0,0,NaN,1.000000,7218,Fine-tuned / drone-capable checkpoint; drone_l...,test_hard_negative,synthetic_distractor_only,180,180
9,rtdetr_l_drone_finetuned,ultralytics_rtdetr,0.50,0,180,NaN,0,0,NaN,1.000000,7218,Fine-tuned / drone-capable checkpoint; drone_l...,test_hard_negative,synthetic_distractor_only,180,180



Corrected recall pivot:


eval_subset                             test_drone_positive  test_mixed
model_name               iou_threshold                                 
grounding_dino           0.25                      0.622222    0.450000
                         0.50                      0.538889    0.372222
rtdetr_l_drone_finetuned 0.25                      0.816667    0.827778
                         0.50                      0.700000    0.694444
yolo11n_drone_finetuned  0.25                      0.727778    0.722222
                         0.50                      0.683333    0.677778


Corrected false-positive image-rate pivot:


eval_subset                             test_hard_negative
model_name               iou_threshold                    
grounding_dino           0.25                     1.000000
                         0.50                     1.000000
rtdetr_l_drone_finetuned 0.25                     1.000000
                         0.50                     1.000000
yolo11n_drone_finetuned  0.25                     0.344444
                         0.50                     0.344444


FP count pivot:


eval_subset                             test_drone_positive  \
model_name               iou_threshold                        
grounding_dino           0.25                             0   
                         0.50                             0   
rtdetr_l_drone_finetuned 0.25                             0   
                         0.50                             0   
yolo11n_drone_finetuned  0.25                             0   
                         0.50                             0   

eval_subset                             test_hard_negative  test_mixed  
model_name               iou_threshold                                  
grounding_dino           0.25                         1873           0  
                         0.50                         1873           0  
rtdetr_l_drone_finetuned 0.25                         7218           0  
                         0.50                         7218           0  
yolo11n_drone_finetuned  0.25                          107           0  
                         0.50                          107           0

In [ ]:
from pathlib import Path
import py_compile

viz_py = PROJECT_ROOT / "code/code/scripts/eval/visualize_predictions.py"
print("Patching:", viz_py)

text = viz_py.read_text()
lines = text.splitlines()

# Find the loop start and the draw function start
start_idx = -1
for i, line in enumerate(lines):
    if "for _, pr in pred_groups.get(image_id, pd.DataFrame()).iterrows():" in line:
        start_idx = i + 1
        break

end_idx = -1
if start_idx != -1:
    for i in range(start_idx, len(lines)):
        if "_draw_xyxy(" in lines[i]:
            end_idx = i
            break

if start_idx != -1 and end_idx != -1:
    new_block = """            prompt = pr.get("prompt", "")
            if not isinstance(prompt, str) or str(prompt).lower() == "nan":
                prompt = pr.get("pred_class_name", "")
            if not isinstance(prompt, str) or str(prompt).lower() == "nan":
                prompt = pr.get("class_name", "")
            if not isinstance(prompt, str) or str(prompt).lower() == "nan":
                prompt = "pred"

            confidence = pr.get("confidence", 0)
            try:
                confidence = float(confidence)
            except Exception:
                confidence = 0.0
            label = f'{str(prompt)[:20]} {confidence:.2f}'"""

    new_lines = lines[:start_idx] + new_block.splitlines() + lines[end_idx:]
    viz_py.write_text("\n".join(new_lines) + "\n")
    print("\u2705 Found the block and replaced it cleanly.")
else:
    print("\u274c Could not find the boundaries for the replacement.")

# Verify syntax after patching
try:
    py_compile.compile(str(viz_py), doraise=True)
    print("\u2705 Syntax verification successful.")
except Exception as e:
    print(f"\u274c Syntax error still exists: {e}")

Patching: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/scripts/eval/visualize_predictions.py
✅ Found the block and replaced it cleanly.
✅ Syntax verification successful.


In [ ]:
# # 14) Metrics + prediction QA by held-out evaluation subset
# # Robust version:
# # - deletes stale empty/corrupt metric files before each eval
# # - supports run_command() returning either int or CompletedProcess
# # - keeps going if one model fails

# import pandas as pd
# from pathlib import Path
# import sys

# models = [
#     "grounding_dino",
#     "yolo11n_drone_finetuned",
#     "rtdetr_l_drone_finetuned",
# ]

# def get_return_code(result):
#     """Support both streaming run_command returning int and subprocess.CompletedProcess."""
#     if isinstance(result, int):
#         return result
#     return getattr(result, "returncode", result)

# def safe_read_csv(path):
#     """Read CSV safely; return empty DataFrame if missing/empty/corrupt."""
#     path = Path(path)
#     if not path.exists():
#         return pd.DataFrame()
#     try:
#         if path.stat().st_size == 0:
#             print("WARNING: CSV exists but is empty:", path)
#             return pd.DataFrame()
#         return pd.read_csv(path)
#     except Exception as e:
#         print("WARNING: failed reading CSV:", path)
#         print("Reason:", repr(e))
#         return pd.DataFrame()

# all_rows = []

# for eval_name, job in EVAL_JOBS.items():
#     print("\n" + "#" * 100)
#     print("METRICS FOR EVAL SUBSET:", eval_name)
#     print("#" * 100)

#     metadata_df = pd.read_csv(job["dataset_root"] / "metadata.csv")

#     n_target_present = (
#         int(metadata_df["target_present"].astype(bool).sum())
#         if "target_present" in metadata_df.columns
#         else 0
#     )

#     n_distractor_only = (
#         int((metadata_df["subset"] == "synthetic_distractor_only").sum())
#         if "subset" in metadata_df.columns
#         else 0
#     )

#     pred_dir = job["output_root"] / "predictions"
#     metrics_dir = job["output_root"] / "metrics"
#     metrics_dir.mkdir(parents=True, exist_ok=True)

#     # Remove stale/corrupt summary files before this subset's evaluation.
#     stale_files = [
#         metrics_dir / "summary_by_model.csv",
#         metrics_dir / "summary_by_model_with_eval_subset.csv",
#     ]
#     for stale in stale_files:
#         if stale.exists():
#             print("Removing stale metrics file:", stale)
#             stale.unlink()

#     subset_rows = []

#     for model in models:
#         pred_csv = pred_dir / f"{model}_predictions.csv"

#         print("\n" + "=" * 80)
#         print("MODEL:", model)
#         print("=" * 80)

#         if not pred_csv.is_file():
#             print("SKIP missing prediction file:", pred_csv)
#             continue

#         pred_df = safe_read_csv(pred_csv)
#         print("prediction rows:", len(pred_df))

#         if len(pred_df) == 0:
#             print("Empty prediction CSV. Adding zero-detection metrics instead of crashing.")

#             for iou_thr in [0.25, 0.50]:
#                 subset_rows.append({
#                     "eval_subset": eval_name,
#                     "source_subset": job["subset"],
#                     "model_name": model,
#                     "model_type": "empty_or_underconfident",
#                     "iou_threshold": iou_thr,
#                     "n_target_present": n_target_present,
#                     "n_distractor_only": n_distractor_only,
#                     "recall": 0.0,
#                     "tp": 0,
#                     "fn": n_target_present,
#                     "mean_best_iou": 0.0,
#                     "false_positive_image_rate": 0.0,
#                     "fp_count": 0,
#                     "note": "No predictions at configured inference threshold; likely undertrained or threshold too high.",
#                 })

#             continue

#         # Important: delete summary before each model too, because evaluator appends/updates it.
#         summary_path = metrics_dir / "summary_by_model.csv"
#         if summary_path.exists():
#             print("Removing previous model summary before eval:", summary_path)
#             summary_path.unlink()

#         result = run_command([
#             sys.executable,
#             "-u",
#             "code/code/scripts/eval/evaluate_detector_predictions.py",
#             "--config", str(job["config"]),
#             "--predictions", str(pred_csv),
#             "--project-root", str(PROJECT_ROOT),
#         ], check=False)

#         rc = get_return_code(result)

#         if rc == 0:
#             summary_df = safe_read_csv(summary_path)

#             if len(summary_df):
#                 summary_df["eval_subset"] = eval_name
#                 summary_df["source_subset"] = job["subset"]
#                 subset_rows.extend(summary_df.to_dict("records"))
#             else:
#                 print("WARNING: evaluator succeeded but summary is missing/empty:", summary_path)

#         else:
#             print("Evaluation failed for this model; see output above.")
#             continue

#         # Visualization / prediction QA. Non-critical.
#         viz_result = run_command([
#             sys.executable,
#             "-u",
#             "code/code/scripts/eval/visualize_predictions.py",
#             "--config", str(job["config"]),
#             "--predictions", str(pred_csv),
#             "--project-root", str(PROJECT_ROOT),
#             "--num-samples", "12",
#         ], check=False)

#         viz_rc = get_return_code(viz_result)
#         if viz_rc != 0:
#             print("WARNING: visualization failed, but metrics may still be valid.")

#     subset_summary = pd.DataFrame(subset_rows)

#     if len(subset_summary):
#         subset_summary = subset_summary.drop_duplicates(
#             subset=["eval_subset", "model_name", "iou_threshold"],
#             keep="last",
#         ).sort_values(["eval_subset", "model_name", "iou_threshold"]).reset_index(drop=True)

#     subset_summary_path = metrics_dir / "summary_by_model_with_eval_subset.csv"
#     subset_summary.to_csv(subset_summary_path, index=False)

#     print("\nSubset summary written to:", subset_summary_path)
#     display(subset_summary)

#     all_rows.extend(subset_summary.to_dict("records"))

# combined_summary = pd.DataFrame(all_rows)

# if len(combined_summary):
#     combined_summary = combined_summary.drop_duplicates(
#         subset=["eval_subset", "model_name", "iou_threshold"],
#         keep="last",
#     ).sort_values(["eval_subset", "model_name", "iou_threshold"]).reset_index(drop=True)

# combined_metrics_dir = EVAL_OUTPUT_ROOT / "combined_metrics"
# combined_metrics_dir.mkdir(parents=True, exist_ok=True)

# combined_path = combined_metrics_dir / "summary_by_model_by_eval_subset.csv"
# combined_summary.to_csv(combined_path, index=False)

# print("\nCombined final-clean summary written to:")
# print(combined_path)
# display(combined_summary)

# if len(combined_summary):
#     print("\nRecall pivot:")
#     display(combined_summary.pivot_table(
#         index=["model_name", "iou_threshold"],
#         columns="eval_subset",
#         values="recall",
#         aggfunc="first",
#     ))

#     print("\nFP count pivot:")
#     display(combined_summary.pivot_table(
#         index=["model_name", "iou_threshold"],
#         columns="eval_subset",
#         values="fp_count",
#         aggfunc="first",
#     ))

#     print("\nFalse-positive image-rate pivot:")
#     display(combined_summary.pivot_table(
#         index=["model_name", "iou_threshold"],
#         columns="eval_subset",
#         values="false_positive_image_rate",
#         aggfunc="first",
#     ))


####################################################################################################
METRICS FOR EVAL SUBSET: test_drone_positive
####################################################################################################
Removing stale metrics file: /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_drone_positive/metrics/summary_by_model.csv

MODEL: grounding_dino
prediction rows: 2061

Running:
/usr/bin/python3 -u scripts/eval/evaluate_detector_predictions.py --config '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs/generated_eval_subsets/eval_test_drone_positive.yaml' --predictions '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_drone_positive/predictions/grounding_dino_predictions.csv' --project-root '/content/drive/MyDrive/Synthetic Stress Testin

""



####################################################################################################
METRICS FOR EVAL SUBSET: test_hard_negative
####################################################################################################

MODEL: grounding_dino
prediction rows: 1873

Running:
/usr/bin/python3 -u scripts/eval/evaluate_detector_predictions.py --config '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs/generated_eval_subsets/eval_test_hard_negative.yaml' --predictions '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_hard_negative/predictions/grounding_dino_predictions.csv' --project-root '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification'

----- live output starts -----
Evaluating grounding_dino from /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_cle

,model_name,model_type,iou_threshold,n_target_present,n_distractor_only,recall,tp,fn,mean_best_iou,false_positive_image_rate,fp_count,note,eval_subset,source_subset
0,grounding_dino,grounding_dino,0.25,0,180,NaN,0,0,NaN,1.0,1873,Open-vocabulary drone prompts; boxes count as ...,test_hard_negative,synthetic_distractor_only
1,grounding_dino,grounding_dino,0.50,0,180,NaN,0,0,NaN,1.0,1873,Open-vocabulary drone prompts; boxes count as ...,test_hard_negative,synthetic_distractor_only



####################################################################################################
METRICS FOR EVAL SUBSET: test_mixed
####################################################################################################

MODEL: grounding_dino
prediction rows: 1999

Running:
/usr/bin/python3 -u scripts/eval/evaluate_detector_predictions.py --config '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/configs/generated_eval_subsets/eval_test_mixed.yaml' --predictions '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_mixed/predictions/grounding_dino_predictions.csv' --project-root '/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification'

----- live output starts -----
Evaluating grounding_dino from /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/evaluation/final_clean_full_curated_v1/test_

...

eval_subset                   test_mixed
model_name     iou_threshold            
grounding_dino 0.25             0.450000
               0.50             0.372222


FP count pivot:


eval_subset                   test_hard_negative  test_mixed
model_name     iou_threshold                                
grounding_dino 0.25                         1873           0
               0.50                         1873           0


False-positive image-rate pivot:


eval_subset                   test_hard_negative
model_name     iou_threshold                    
grounding_dino 0.25                          1.0
               0.50                          1.0

In [ ]:
# 15) Optional: YOLO confidence diagnostic on one positive test image
# Use this if YOLO predictions are empty or unexpectedly weak.

from pathlib import Path

try:
    from ultralytics import YOLO

    weights = PROJECT_ROOT / "outputs/training/yolo" / YOLO_RUN_NAME / "weights/best.pt"

    # Pick first test positive image if available.
    pos_root = EVAL_JOBS["test_drone_positive"]["dataset_root"]
    pos_meta = pd.read_csv(pos_root / "metadata.csv")
    sample_image_id = pos_meta["image_id"].astype(str).iloc[0]
    image = pos_root / "images" / sample_image_id

    print("weights exists:", weights.exists(), weights)
    print("sample image exists:", image.exists(), image)

    if weights.exists() and image.exists():
        model = YOLO(str(weights))
        for conf in [0.25, 0.10, 0.05, 0.02, 0.01, 0.005, 0.001]:
            results = model.predict(
                source=str(image),
                conf=conf,
                imgsz=640,
                device=0,
                verbose=False,
                max_det=300,
            )
            n_boxes = len(results[0].boxes)
            if n_boxes:
                confs = results[0].boxes.conf.cpu().numpy()
                print(f"conf={conf}: boxes={n_boxes}, max_conf={confs.max():.5f}, min_conf={confs.min():.5f}")
            else:
                print(f"conf={conf}: boxes=0")

except Exception as e:
    print("YOLO confidence diagnostic failed:", repr(e))


weights exists: True /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/outputs/training/yolo/yolo11n_drone_colab/weights/best.pt
sample image exists: True /content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/data/evaluation/final_clean_full_curated_v1/test_drone_positive/images/img_001276.png
conf=0.25: boxes=2, max_conf=0.45179, min_conf=0.28569
conf=0.1: boxes=2, max_conf=0.45179, min_conf=0.28569
conf=0.05: boxes=3, max_conf=0.45179, min_conf=0.06056
conf=0.02: boxes=4, max_conf=0.45179, min_conf=0.02124
conf=0.01: boxes=5, max_conf=0.45179, min_conf=0.01401
conf=0.005: boxes=7, max_conf=0.45179, min_conf=0.00538
conf=0.001: boxes=20, max_conf=0.45179, min_conf=0.00103


In [ ]:
# 16) Save final-clean results zip to Drive
results_dir = PROJECT_ROOT / "drone_colab_results"
results_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_base = results_dir / f"drone_colab_final_clean_results_{timestamp}"

archive_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(EVAL_OUTPUT_ROOT),
)

print("Saved results archive:")
print(archive_path)


Saved results archive:
/content/drive/MyDrive/Synthetic Stress Testing for Skyborne Drone Identification/drone_colab_results/drone_colab_final_clean_results_20260628_172157.zip


## Notes

This notebook implements the final clean subset evaluation protocol:

- Train YOLO/RT-DETR on the **train split containing all three conditions**:
  - `synthetic_drone_positive`
  - `synthetic_distractor_only`
  - `synthetic_drone_plus_distractor`

- Evaluate GroundingDINO, YOLO, and RT-DETR separately on held-out test subsets:
  - `test_drone_positive`
  - `test_hard_negative`
  - `test_mixed`

- The main final table is saved to:

`outputs/evaluation/final_clean_full_curated_v1/combined_metrics/summary_by_model_by_eval_subset.csv`

Scientific interpretation:

- For `test_drone_positive` and `test_mixed`, recall and IoU metrics are meaningful.
- For `test_hard_negative`, recall is not meaningful because no drone exists. Focus on false-positive image rate and FP count.
- Hard negatives are included in training as negative examples, not as a standalone detector-training dataset.
